## Part 4.1: Initialise PostgreSQL Schemas, Tables, Constraints, and Load Session

### Objective

Prepare the PostgreSQL database structures required for controlled Cyclistic data loading.

Parts 1–3 validated the environment, established the approved schema contract, and completed the data-quality audit. Part 4 begins database processing.

This section creates database structures only. It does not load CSV records into staging or quarantine.

### Database Architecture

The initial database-processing layer uses three PostgreSQL schemas:

```text
stg
quarantine
audit
```

Their responsibilities are:

| Schema       | Responsibility                                                                                           |
| ------------ | -------------------------------------------------------------------------------------------------------- |
| `stg`        | Stores source records that pass failure-severity quality controls and remain eligible for transformation |
| `quarantine` | Stores source records that violate failure-severity rules                                                |
| `audit`      | Stores database-load execution history and reconciliation information                                    |

PostgreSQL schemas must exist before schema-qualified tables can be created.

The pipeline therefore creates the schemas explicitly:

```sql
CREATE SCHEMA IF NOT EXISTS stg;
CREATE SCHEMA IF NOT EXISTS quarantine;
CREATE SCHEMA IF NOT EXISTS audit;
```

PostgreSQL and Azure Database for PostgreSQL do not automatically convert:

```text
stg.cyclistic_rides
```

into:

```text
stg_cyclistic_rides
```

The first form refers to table `cyclistic_rides` inside schema `stg`. The second form is a separate table name in the current schema.

### Staging Table

`stg.cyclistic_rides` stores records eligible for downstream processing.

It contains the approved Cyclistic source fields:

* `ride_id`;
* `rideable_type`;
* `started_at`;
* `ended_at`;
* station names and identifiers;
* start and end coordinates;
* `member_casual`.

It also adds technical lineage fields:

* `load_session_id`;
* `quality_audit_session_id`;
* `source_file`;
* `source_row_number`;
* `source_record_key`;
* `ride_duration_seconds`;
* `has_quality_warning`;
* `warning_rule_ids`;
* `loaded_at`.

The source record key uniquely identifies a physical source row within one load session.

### Quarantine Table

`quarantine.cyclistic_rides` preserves records that violate failure-severity quality rules.

It stores:

* the original source values;
* source filename and row number;
* database-load and quality-audit session IDs;
* failed rule IDs;
* quarantine reason;
* optional warning rule IDs;
* quarantine timestamp.

The initial expected quarantine condition is:

```text
DQ007_END_AFTER_START
```

for the 29 records in `202511-divvy-tripdata.csv` whose ending timestamp is not later than their starting timestamp.

No source record is deleted or corrected in this section.

### Pipeline Job Log

`audit.pipeline_job_log` records one row for each database-load execution.

The initial record is inserted with status:

```text
INITIALISED
```

Later Part 4 sections will update the same record through states such as:

```text
LOADING
VALIDATING
COMPLETED
FAILED
```

The table records:

* load session ID;
* related quality-audit session ID;
* schema and ruleset versions;
* source file and row counts;
* staging and quarantine counts;
* rejected and warning counts;
* execution timestamps;
* error information;
* execution metadata.

### Transaction Policy

Schema creation, table creation, index creation, validation, and load-session registration are executed in a PostgreSQL transaction.

If a database operation fails:

* the transaction is rolled back;
* no partial job-log record is committed;
* the failure is written to `pipeline_audit.log`;
* the notebook raises the original error.

### Idempotency Policy

The database objects use:

```sql
CREATE SCHEMA IF NOT EXISTS
CREATE TABLE IF NOT EXISTS
CREATE INDEX IF NOT EXISTS
```

This allows the setup cell to be rerun safely.

However, `IF NOT EXISTS` does not prove that an existing table has the correct columns. The section therefore validates required table columns after creation and stops if an incompatible existing table is detected.

### Expected Outcome

* PostgreSQL connectivity is reverified.
* The current database and role are identified.
* Schema-creation permission is confirmed.
* `stg`, `quarantine`, and `audit` exist.
* Required tables and indexes exist.
* Existing table structures contain all required columns.
* A unique database-load session is generated.
* The load session is registered as `INITIALISED`.
* Part 4.2 can begin controlled chunked loading and row routing.


In [ ]:
# ==========================================
# Part 4.1: Initialise PostgreSQL Schemas
# ==========================================

from datetime import datetime
from pathlib import Path
from uuid import uuid4

import psycopg2
from psycopg2 import OperationalError, sql


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_4_1_VALUES = {
    "DB_HOST": DB_HOST,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_PORT": DB_PORT,
    "DB_SSLMODE": DB_SSLMODE,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
}

missing_part_4_1_values = [
    variable_name
    for variable_name, variable_value
    in REQUIRED_PART_4_1_VALUES.items()
    if (
        variable_value is None
        or (
            isinstance(variable_value, str)
            and not variable_value.strip()
        )
    )
]

if missing_part_4_1_values:
    raise RuntimeError(
        "Part 4.1 is missing required value(s): "
        + ", ".join(
            sorted(missing_part_4_1_values)
        )
    )


if not isinstance(
    PIPELINE_AUDIT_PATH,
    Path,
):
    raise TypeError(
        "PIPELINE_AUDIT_PATH must be a pathlib.Path object."
    )


if not PIPELINE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        "Pipeline audit log does not exist:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


if not PIPELINE_AUDIT_PATH.is_file():
    raise FileExistsError(
        "PIPELINE_AUDIT_PATH exists but is not a file:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


# ------------------------------------------
# Initialise database-processing session
# ------------------------------------------

# Reuse this identifier throughout Parts 4.1–4.5.
DATABASE_PROCESSING_SESSION_ID = (
    uuid4().hex[:8]
)

DATABASE_PROCESSING_START_TIME = (
    datetime.now(
        MELBOURNE_TIMEZONE
    )
)

database_run_warnings = []
database_run_errors = []


SUPPORTED_DATABASE_LOG_LEVELS = {
    "INFO",
    "WARNING",
    "ERROR",
    "CRITICAL",
}


def log_database_event(
    message: str,
    level: str = "INFO",
) -> str:
    """
    Append one database-processing event to the shared pipeline
    audit log and print it in the notebook.
    """
    normalised_level = level.upper().strip()

    if (
        normalised_level
        not in SUPPORTED_DATABASE_LOG_LEVELS
    ):
        raise ValueError(
            "Unsupported database log level: "
            f"{level}"
        )

    if (
        not isinstance(message, str)
        or not message.strip()
    ):
        raise ValueError(
            "Database log message must be a non-empty string."
        )

    timestamp = datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds")

    log_entry = (
        f"[{timestamp}] "
        f"[DB_RUN:{DATABASE_PROCESSING_SESSION_ID}] "
        f"[{normalised_level}] "
        f"{message.strip()}"
    )

    with PIPELINE_AUDIT_PATH.open(
        mode="a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(
            log_entry + "\n"
        )

    print(log_entry)

    event_record = {
        "timestamp": timestamp,
        "database_processing_session_id": (
            DATABASE_PROCESSING_SESSION_ID
        ),
        "level": normalised_level,
        "message": message.strip(),
    }

    if normalised_level == "WARNING":
        database_run_warnings.append(
            event_record
        )

    elif normalised_level in {
        "ERROR",
        "CRITICAL",
    }:
        database_run_errors.append(
            event_record
        )

    return log_entry


# ------------------------------------------
# Define required PostgreSQL schemas
# ------------------------------------------

REQUIRED_DATABASE_SCHEMAS = (
    "stg",
    "quarantine",
    "audit",
)


# ------------------------------------------
# Open transaction and initialise schemas
# ------------------------------------------

connection = None
cursor = None

schemas_existing_before = []
schemas_created_this_run = []
verified_schemas = []


log_database_event(
    message=(
        "DATABASE_PROCESSING_SESSION_START: "
        "PostgreSQL database processing initialised."
    ),
    level="INFO",
)


try:
    connection = psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        sslmode=DB_SSLMODE,
        connect_timeout=10,
    )

    connection.autocommit = False
    cursor = connection.cursor()


    # --------------------------------------
    # Confirm connection identity
    # --------------------------------------

    cursor.execute(
        """
        SELECT
            current_database(),
            current_user;
        """
    )

    (
        connected_database,
        connected_user,
    ) = cursor.fetchone()


    if connected_database != DB_NAME:
        raise RuntimeError(
            "Connected database does not match DB_NAME. "
            f"Expected={DB_NAME}; "
            f"Observed={connected_database}."
        )


    log_database_event(
        message=(
            "DATABASE_CONNECTION_OPENED: "
            f"database={connected_database}; "
            f"user={connected_user}; "
            f"host={DB_HOST}; "
            f"port={DB_PORT}."
        ),
        level="INFO",
    )


    # --------------------------------------
    # Inspect schemas before initialisation
    # --------------------------------------

    cursor.execute(
        """
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name = ANY(%s)
        ORDER BY schema_name;
        """,
        (list(REQUIRED_DATABASE_SCHEMAS),),
    )

    schemas_existing_before = [
        row[0]
        for row in cursor.fetchall()
    ]


    schemas_missing_before = [
        schema_name
        for schema_name
        in REQUIRED_DATABASE_SCHEMAS
        if schema_name
        not in schemas_existing_before
    ]


    # --------------------------------------
    # Validate database CREATE privilege
    # --------------------------------------

    cursor.execute(
        """
        SELECT has_database_privilege(
            current_user,
            current_database(),
            'CREATE'
        );
        """
    )

    can_create_schema = bool(
        cursor.fetchone()[0]
    )


    if (
        schemas_missing_before
        and not can_create_schema
    ):
        raise PermissionError(
            "The connected PostgreSQL role cannot create "
            "schemas in the current database. "
            "Missing schema(s): "
            + ", ".join(
                schemas_missing_before
            )
        )


    # --------------------------------------
    # Create schemas idempotently
    # --------------------------------------

    for schema_name in (
        REQUIRED_DATABASE_SCHEMAS
    ):
        create_schema_statement = (
            sql.SQL(
                "CREATE SCHEMA IF NOT EXISTS {}"
            ).format(
                sql.Identifier(schema_name)
            )
        )

        cursor.execute(
            create_schema_statement
        )


        if schema_name in schemas_existing_before:
            log_database_event(
                message=(
                    "DATABASE_SCHEMA_FOUND: "
                    f"schema={schema_name}."
                ),
                level="INFO",
            )

        else:
            schemas_created_this_run.append(
                schema_name
            )

            log_database_event(
                message=(
                    "DATABASE_SCHEMA_CREATED: "
                    f"schema={schema_name}."
                ),
                level="INFO",
            )


    # --------------------------------------
    # Verify schema existence and ownership
    # --------------------------------------

    cursor.execute(
        """
        SELECT
            namespace.nspname AS schema_name,
            owner_role.rolname AS schema_owner
        FROM pg_namespace AS namespace
        INNER JOIN pg_roles AS owner_role
            ON owner_role.oid = namespace.nspowner
        WHERE namespace.nspname = ANY(%s)
        ORDER BY namespace.nspname;
        """,
        (list(REQUIRED_DATABASE_SCHEMAS),),
    )

    verified_schema_rows = (
        cursor.fetchall()
    )

    verified_schema_metadata = {
        schema_name: {
            "owner": schema_owner,
        }
        for (
            schema_name,
            schema_owner,
        ) in verified_schema_rows
    }


    missing_after_initialisation = [
        schema_name
        for schema_name
        in REQUIRED_DATABASE_SCHEMAS
        if schema_name
        not in verified_schema_metadata
    ]


    if missing_after_initialisation:
        raise RuntimeError(
            "Required schema verification failed. "
            "Missing schema(s): "
            + ", ".join(
                missing_after_initialisation
            )
        )


    # --------------------------------------
    # Verify schema privileges
    # --------------------------------------

    for schema_name in (
        REQUIRED_DATABASE_SCHEMAS
    ):
        cursor.execute(
            """
            SELECT
                has_schema_privilege(
                    current_user,
                    %s,
                    'USAGE'
                ),
                has_schema_privilege(
                    current_user,
                    %s,
                    'CREATE'
                );
            """,
            (
                schema_name,
                schema_name,
            ),
        )

        (
            has_usage_privilege,
            has_create_privilege,
        ) = cursor.fetchone()


        if not has_usage_privilege:
            raise PermissionError(
                "The connected PostgreSQL role does not have "
                f"USAGE privilege on schema '{schema_name}'."
            )


        if not has_create_privilege:
            raise PermissionError(
                "The connected PostgreSQL role does not have "
                f"CREATE privilege on schema '{schema_name}'."
            )


        verified_schemas.append(
            {
                "schema_name": schema_name,
                "owner": (
                    verified_schema_metadata[
                        schema_name
                    ]["owner"]
                ),
                "usage_privilege": bool(
                    has_usage_privilege
                ),
                "create_privilege": bool(
                    has_create_privilege
                ),
            }
        )

        log_database_event(
            message=(
                "DATABASE_SCHEMA_VERIFIED: "
                f"schema={schema_name}; "
                f"owner="
                f"{verified_schema_metadata[schema_name]['owner']}; "
                "usage_privilege=True; "
                "create_privilege=True."
            ),
            level="INFO",
        )


    # --------------------------------------
    # Commit schema initialisation
    # --------------------------------------

    connection.commit()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_COMMITTED: "
            f"created={len(schemas_created_this_run)}; "
            f"existing={len(schemas_existing_before)}; "
            f"verified={len(verified_schemas)}."
        ),
        level="INFO",
    )


except OperationalError as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_FAILED: "
            "Unable to connect to PostgreSQL. "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise ConnectionError(
        "Unable to open the PostgreSQL connection required "
        "for Part 4.1."
    ) from error


except (
    PermissionError,
    RuntimeError,
    psycopg2.DatabaseError,
) as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_ROLLED_BACK: "
            f"error_type={type(error).__name__}; "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise


finally:
    if cursor is not None:
        cursor.close()
        print("✓ Database cursor closed.")

    if connection is not None:
        connection.close()
        print("✓ Database connection closed.")


# ------------------------------------------
# Build Part 4.1 summary
# ------------------------------------------

DATABASE_SCHEMA_INITIALISATION_SUMMARY = {
    "database_processing_session_id": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "database": connected_database,
    "connected_user": connected_user,
    "required_schemas": list(
        REQUIRED_DATABASE_SCHEMAS
    ),
    "schemas_existing_before": list(
        schemas_existing_before
    ),
    "schemas_created_this_run": list(
        schemas_created_this_run
    ),
    "verified_schemas": list(
        verified_schemas
    ),
    "status": "SUCCESS",
    "completed_at": datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds"),
}


# ------------------------------------------
# Display final summary
# ------------------------------------------

print(
    "\n========== PostgreSQL Schema Initialisation "
    "==========\n"
)

print(
    f"Database-processing session : "
    f"{DATABASE_PROCESSING_SESSION_ID}"
)
print(
    f"Database                    : "
    f"{connected_database}"
)
print(
    f"Connected role              : "
    f"{connected_user}"
)
print(
    f"Required schemas            : "
    f"{len(REQUIRED_DATABASE_SCHEMAS)}"
)
print(
    f"Schemas created             : "
    f"{len(schemas_created_this_run)}"
)
print(
    f"Schemas already existing    : "
    f"{len(schemas_existing_before)}"
)
print(
    f"Schemas verified            : "
    f"{len(verified_schemas)}"
)


print("\nSchema details:")

for schema_record in verified_schemas:
    print(
        f"  {schema_record['schema_name']:<12} "
        f"owner={schema_record['owner']:<20} "
        "USAGE=True CREATE=True"
    )


print("\n" + "=" * 75)
print("✓ Required PostgreSQL schemas exist.")
print("✓ Existing schemas were preserved.")
print("✓ Missing schemas were created idempotently.")
print("✓ Schema permissions verified.")
print("✓ Schema initialisation transaction committed.")
print("✓ Part 4.1 completed successfully.")
print("=" * 75)

## Part 4.2: Provision and Validate PostgreSQL Tables

### Objective

Create and validate the PostgreSQL tables required for controlled Cyclistic data loading.

Part 4.1 established the database namespaces:

```text
stg
quarantine
audit
```

Part 4.2 provisions the following tables:

```text
stg.cyclistic_rides
quarantine.cyclistic_rides
audit.pipeline_job_log
```

This section creates database infrastructure only. It does not read CSV files or insert source records.

### Approved Schema Dependency

The staging table must be based on the approved schema stored in:

```text
master_schema.json
```

Before table provisioning begins, this section reloads and verifies:

* `active_schema_status` is `APPROVED`;
* `active_schema` is not empty;
* `active_schema_version` is populated;
* all approved SQL types match a controlled type allowlist.

The approved schema is never inferred or modified during Part 4.2.

### Staging Table

The staging table contains the approved Cyclistic columns plus operational lineage fields:

```text
staging_record_id
source_file
source_row_number
pipeline_session_id
quality_audit_session_id
loaded_at
```

`staging_record_id` is the technical primary key.

`ride_id` receives a unique index as a defensive database control because the approved quality contract requires it to be unique across the source dataset.

The lineage fields allow each loaded row to be traced back to:

* its source CSV;
* its source row number;
* the database-processing session;
* the quality-audit session.

### Quarantine Table

The quarantine table stores records that violate failure-severity quality rules.

It contains:

```text
quarantine_id
ride_id
failed_rule
reason
source_file
source_row_number
raw_record
pipeline_session_id
quality_audit_session_id
created_at
```

`raw_record` uses PostgreSQL `JSONB` so the complete source record can be preserved without requiring the quarantine table to duplicate every approved schema column.

One source row may eventually produce more than one quarantine finding when multiple failure rules are violated.

### Pipeline Job Log

`audit.pipeline_job_log` records each source-file processing attempt.

It includes:

* job ID;
* pipeline session ID;
* quality-audit session ID;
* source filename;
* SHA-256 file checksum;
* status;
* start and completion timestamps;
* rows read;
* rows loaded;
* rows quarantined;
* duration;
* approved schema version;
* quality-ruleset version;
* error details.

A partial unique index is created for successful loads:

```text
source_file + file_checksum
```

This supports the agreed idempotency policy:

```text
If the same file and checksum already completed successfully,
skip it unless a future force-reload option is enabled.
```

Failed attempts may still be recorded multiple times.

### Existing-Table Policy

`CREATE TABLE IF NOT EXISTS` prevents accidental replacement of existing tables, but it does not verify their definitions.

After provisioning, Part 4.2 inspects PostgreSQL metadata and validates:

* required columns;
* SQL data types;
* nullability;
* primary keys;
* named indexes;
* required constraints.

If an existing table differs from the expected contract, the transaction is rolled back and the table is not silently altered.

Schema migration and controlled table alteration should be handled explicitly rather than hidden inside normal provisioning.

### Transaction Policy

All table and index provisioning occurs in one transaction.

```text
Open connection
↓
Reload approved schema
↓
Create missing tables
↓
Create missing indexes
↓
Inspect live table metadata
↓
Validate columns and constraints
↓
Commit
```

If validation fails:

```text
Rollback
↓
Log the mismatch
↓
Close the connection
```

### Expected Outcome

* All three required tables exist.
* The staging table matches the approved schema plus lineage columns.
* The quarantine table supports complete row preservation and failure lineage.
* The job-log table supports file-level execution history and idempotency.
* Primary keys, constraints, and indexes are validated.
* No source records are loaded.
* Part 4 is ready for load-session initialisation and idempotency checks.


In [ ]:
# ==========================================
# Part 4.2: Provision and Validate
# PostgreSQL Tables
# ==========================================

import json
import re
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Any

import psycopg2
from psycopg2 import OperationalError, sql


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_4_2_VALUES = {
    "DATABASE_PROCESSING_SESSION_ID": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "DB_HOST": DB_HOST,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_PORT": DB_PORT,
    "DB_SSLMODE": DB_SSLMODE,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "QUALITY_RULES_PATH": QUALITY_RULES_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
}

missing_part_4_2_values = [
    variable_name
    for variable_name, variable_value
    in REQUIRED_PART_4_2_VALUES.items()
    if (
        variable_value is None
        or (
            isinstance(variable_value, str)
            and not variable_value.strip()
        )
    )
]

if missing_part_4_2_values:
    raise RuntimeError(
        "Part 4.2 is missing required value(s): "
        + ", ".join(
            sorted(missing_part_4_2_values)
        )
    )


for path_name in (
    "MASTER_SCHEMA_PATH",
    "QUALITY_RULES_PATH",
):
    path_value = REQUIRED_PART_4_2_VALUES[
        path_name
    ]

    if not isinstance(path_value, Path):
        raise TypeError(
            f"{path_name} must be a pathlib.Path object."
        )

    if not path_value.exists():
        raise FileNotFoundError(
            f"{path_name} does not exist:\n"
            f"{path_value}"
        )

    if not path_value.is_file():
        raise FileExistsError(
            f"{path_name} exists but is not a file:\n"
            f"{path_value}"
        )


# ------------------------------------------
# Reload approved schema and quality metadata
# ------------------------------------------

def load_required_json_object(
    file_path: Path,
    object_name: str,
) -> dict:
    """
    Load a required JSON object without using stale notebook state.
    """
    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
        ) as json_file:
            loaded_data = json.load(json_file)

    except json.JSONDecodeError as error:
        raise ValueError(
            f"{object_name} contains invalid JSON."
        ) from error

    if not isinstance(loaded_data, dict):
        raise TypeError(
            f"{object_name} must contain a JSON object."
        )

    return loaded_data


database_master_registry = (
    load_required_json_object(
        file_path=MASTER_SCHEMA_PATH,
        object_name="master_schema.json",
    )
)

database_quality_rules = (
    load_required_json_object(
        file_path=QUALITY_RULES_PATH,
        object_name="quality_rules.json",
    )
)


DATABASE_ACTIVE_SCHEMA = (
    database_master_registry.get(
        "active_schema",
        {},
    )
)

DATABASE_ACTIVE_SCHEMA_STATUS = (
    database_master_registry.get(
        "active_schema_status"
    )
)

DATABASE_ACTIVE_SCHEMA_VERSION = (
    database_master_registry.get(
        "active_schema_version"
    )
)

DATABASE_QUALITY_RULESET_VERSION = (
    database_quality_rules.get(
        "metadata",
        {},
    ).get(
        "ruleset_version"
    )
)


if DATABASE_ACTIVE_SCHEMA_STATUS != "APPROVED":
    raise RuntimeError(
        "Part 4.2 requires active_schema_status=APPROVED."
    )


if (
    not isinstance(
        DATABASE_ACTIVE_SCHEMA,
        dict,
    )
    or not DATABASE_ACTIVE_SCHEMA
):
    raise RuntimeError(
        "Part 4.2 requires a non-empty active_schema."
    )


if (
    not isinstance(
        DATABASE_ACTIVE_SCHEMA_VERSION,
        str,
    )
    or not DATABASE_ACTIVE_SCHEMA_VERSION.strip()
):
    raise RuntimeError(
        "Part 4.2 requires a valid active_schema_version."
    )


if (
    not isinstance(
        DATABASE_QUALITY_RULESET_VERSION,
        str,
    )
    or not DATABASE_QUALITY_RULESET_VERSION.strip()
):
    raise RuntimeError(
        "Part 4.2 requires a valid quality ruleset version."
    )


# ------------------------------------------
# Validate approved SQL type definitions
# ------------------------------------------

ALLOWED_SQL_TYPE_PATTERNS = (
    re.compile(
        r"^VARCHAR\([1-9][0-9]*\)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^NUMERIC\([1-9][0-9]*,\s*[0-9]+\)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(SMALLINT|INTEGER|BIGINT)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(REAL|DOUBLE PRECISION)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(BOOLEAN|DATE|TEXT|UUID|JSONB)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^TIMESTAMP"
        r"( WITHOUT TIME ZONE| WITH TIME ZONE)?$",
        re.IGNORECASE,
    ),
)


def normalise_declared_sql_type(
    sql_type_value: str,
) -> str:
    """
    Normalise whitespace and capitalisation in an approved SQL type.
    """
    if not isinstance(
        sql_type_value,
        str,
    ):
        raise TypeError(
            "Approved SQL type values must be strings."
        )

    normalised_type = re.sub(
        r"\s+",
        " ",
        sql_type_value.strip().upper(),
    )

    if not any(
        pattern.fullmatch(normalised_type)
        for pattern in ALLOWED_SQL_TYPE_PATTERNS
    ):
        raise ValueError(
            "Unsupported or unsafe approved SQL type: "
            f"{sql_type_value}"
        )

    return normalised_type


NORMALISED_ACTIVE_SCHEMA = {
    column_name: normalise_declared_sql_type(
        sql_type_value
    )
    for column_name, sql_type_value
    in DATABASE_ACTIVE_SCHEMA.items()
}


# ------------------------------------------
# Protect reserved lineage column names
# ------------------------------------------

STAGING_LINEAGE_COLUMNS = {
    "staging_record_id",
    "source_file",
    "source_row_number",
    "pipeline_session_id",
    "quality_audit_session_id",
    "loaded_at",
}


lineage_name_conflicts = (
    STAGING_LINEAGE_COLUMNS
    & set(NORMALISED_ACTIVE_SCHEMA)
)

if lineage_name_conflicts:
    raise ValueError(
        "Approved schema conflicts with staging lineage column(s): "
        + ", ".join(
            sorted(lineage_name_conflicts)
        )
    )


if "ride_id" not in NORMALISED_ACTIVE_SCHEMA:
    raise ValueError(
        "Approved schema must contain ride_id."
    )


# ------------------------------------------
# Define expected table contracts
# ------------------------------------------

def column_contract(
    sql_type: str,
    nullable: bool,
) -> dict:
    return {
        "sql_type": sql_type,
        "nullable": nullable,
    }


EXPECTED_STAGING_COLUMNS = {
    "staging_record_id": column_contract(
        "UUID",
        False,
    ),
    **{
        column_name: column_contract(
            sql_type_value,
            False
            if column_name == "ride_id"
            else True,
        )
        for column_name, sql_type_value
        in NORMALISED_ACTIVE_SCHEMA.items()
    },
    "source_file": column_contract(
        "VARCHAR(255)",
        False,
    ),
    "source_row_number": column_contract(
        "BIGINT",
        False,
    ),
    "pipeline_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "quality_audit_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "loaded_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
}


EXPECTED_QUARANTINE_COLUMNS = {
    "quarantine_id": column_contract(
        "UUID",
        False,
    ),
    "ride_id": column_contract(
        "VARCHAR(255)",
        True,
    ),
    "failed_rule": column_contract(
        "VARCHAR(100)",
        False,
    ),
    "reason": column_contract(
        "TEXT",
        False,
    ),
    "source_file": column_contract(
        "VARCHAR(255)",
        False,
    ),
    "source_row_number": column_contract(
        "BIGINT",
        False,
    ),
    "raw_record": column_contract(
        "JSONB",
        False,
    ),
    "pipeline_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "quality_audit_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "created_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
}


EXPECTED_JOB_LOG_COLUMNS = {
    "job_id": column_contract(
        "UUID",
        False,
    ),
    "pipeline_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "quality_audit_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "source_file": column_contract(
        "VARCHAR(255)",
        False,
    ),
    "file_checksum": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "status": column_contract(
        "VARCHAR(30)",
        False,
    ),
    "started_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
    "completed_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        True,
    ),
    "rows_read": column_contract(
        "BIGINT",
        False,
    ),
    "rows_loaded": column_contract(
        "BIGINT",
        False,
    ),
    "rows_quarantined": column_contract(
        "BIGINT",
        False,
    ),
    "duration_seconds": column_contract(
        "NUMERIC(18, 3)",
        True,
    ),
    "schema_version": column_contract(
        "VARCHAR(50)",
        False,
    ),
    "quality_rules_version": column_contract(
        "VARCHAR(50)",
        False,
    ),
    "error_message": column_contract(
        "TEXT",
        True,
    ),
    "created_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
    "updated_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
}


EXPECTED_TABLE_CONTRACTS = {
    ("audit", "pipeline_job_log"): {
        "columns": EXPECTED_JOB_LOG_COLUMNS,
        "primary_key": ["job_id"],
        "required_constraints": {
            "ck_pipeline_job_log_status",
            "ck_pipeline_job_log_counts",
        },
        "required_indexes": {
            "ux_pipeline_job_log_success_file",
            "ix_pipeline_job_log_session",
            "ix_pipeline_job_log_source_file",
        },
    },
    ("stg", "cyclistic_rides"): {
        "columns": EXPECTED_STAGING_COLUMNS,
        "primary_key": [
            "staging_record_id"
        ],
        "required_constraints": set(),
        "required_indexes": {
            "ux_stg_cyclistic_rides_ride_id",
            "ix_stg_cyclistic_rides_source",
            "ix_stg_cyclistic_rides_session",
        },
    },
    ("quarantine", "cyclistic_rides"): {
        "columns": EXPECTED_QUARANTINE_COLUMNS,
        "primary_key": [
            "quarantine_id"
        ],
        "required_constraints": set(),
        "required_indexes": {
            "ix_quarantine_cyclistic_rides_source",
            "ix_quarantine_cyclistic_rides_rule",
            "ix_quarantine_cyclistic_rides_session",
        },
    },
}


# ------------------------------------------
# PostgreSQL metadata normalisation
# ------------------------------------------

def canonical_database_type(
    data_type: str,
    character_maximum_length: int | None,
    numeric_precision: int | None,
    numeric_scale: int | None,
) -> str:
    """
    Convert information_schema type metadata into a canonical form.
    """
    normalised_data_type = (
        data_type.strip().upper()
    )

    if normalised_data_type in {
        "CHARACTER VARYING",
        "VARCHAR",
    }:
        return (
            f"VARCHAR("
            f"{character_maximum_length}"
            f")"
        )

    if normalised_data_type in {
        "NUMERIC",
        "DECIMAL",
    }:
        return (
            f"NUMERIC("
            f"{numeric_precision}, "
            f"{numeric_scale}"
            f")"
        )

    if normalised_data_type in {
        "TIMESTAMP WITHOUT TIME ZONE",
        "TIMESTAMP WITH TIME ZONE",
    }:
        return normalised_data_type

    return normalised_data_type


def normalise_type_spacing(
    sql_type_value: str,
) -> str:
    """
    Normalise spacing around commas for reliable comparisons.
    """
    normalised_value = re.sub(
        r"\s+",
        " ",
        sql_type_value.strip().upper(),
    )

    normalised_value = re.sub(
        r"\s*,\s*",
        ", ",
        normalised_value,
    )

    if normalised_value == "TIMESTAMP":
        return "TIMESTAMP WITHOUT TIME ZONE"

    return normalised_value


# ------------------------------------------
# Generate staging table SQL safely
# ------------------------------------------

def build_staging_table_statement():
    """
    Build the staging table from the approved schema and lineage
    contract.
    """
    column_definitions = [
        sql.SQL(
            "{} UUID NOT NULL"
        ).format(
            sql.Identifier(
                "staging_record_id"
            )
        )
    ]

    for column_name, sql_type_value in (
        NORMALISED_ACTIVE_SCHEMA.items()
    ):
        nullability_sql = (
            sql.SQL(" NOT NULL")
            if column_name == "ride_id"
            else sql.SQL("")
        )

        column_definitions.append(
            sql.SQL(
                "{} {}{}"
            ).format(
                sql.Identifier(column_name),
                sql.SQL(sql_type_value),
                nullability_sql,
            )
        )

    column_definitions.extend(
        [
            sql.SQL(
                "{} VARCHAR(255) NOT NULL"
            ).format(
                sql.Identifier(
                    "source_file"
                )
            ),
            sql.SQL(
                "{} BIGINT NOT NULL"
            ).format(
                sql.Identifier(
                    "source_row_number"
                )
            ),
            sql.SQL(
                "{} VARCHAR(64) NOT NULL"
            ).format(
                sql.Identifier(
                    "pipeline_session_id"
                )
            ),
            sql.SQL(
                "{} VARCHAR(64) NOT NULL"
            ).format(
                sql.Identifier(
                    "quality_audit_session_id"
                )
            ),
            sql.SQL(
                "{} TIMESTAMPTZ NOT NULL "
                "DEFAULT CURRENT_TIMESTAMP"
            ).format(
                sql.Identifier(
                    "loaded_at"
                )
            ),
            sql.SQL(
                "CONSTRAINT {} PRIMARY KEY ({})"
            ).format(
                sql.Identifier(
                    "pk_stg_cyclistic_rides"
                ),
                sql.Identifier(
                    "staging_record_id"
                ),
            ),
        ]
    )

    return sql.SQL(
        "CREATE TABLE IF NOT EXISTS {}.{} ({})"
    ).format(
        sql.Identifier("stg"),
        sql.Identifier(
            "cyclistic_rides"
        ),
        sql.SQL(", ").join(
            column_definitions
        ),
    )


# ------------------------------------------
# Table validation helpers
# ------------------------------------------

def fetch_table_columns(
    cursor,
    schema_name: str,
    table_name: str,
) -> dict:
    """
    Read live column metadata from information_schema.
    """
    cursor.execute(
        """
        SELECT
            column_name,
            data_type,
            character_maximum_length,
            numeric_precision,
            numeric_scale,
            is_nullable
        FROM information_schema.columns
        WHERE
            table_schema = %s
            AND table_name = %s
        ORDER BY ordinal_position;
        """,
        (
            schema_name,
            table_name,
        ),
    )

    live_columns = {}

    for (
        column_name,
        data_type,
        character_maximum_length,
        numeric_precision,
        numeric_scale,
        is_nullable,
    ) in cursor.fetchall():

        live_columns[column_name] = {
            "sql_type": canonical_database_type(
                data_type=(
                    data_type
                ),
                character_maximum_length=(
                    character_maximum_length
                ),
                numeric_precision=(
                    numeric_precision
                ),
                numeric_scale=(
                    numeric_scale
                ),
            ),
            "nullable": (
                is_nullable == "YES"
            ),
        }

    return live_columns


def fetch_primary_key_columns(
    cursor,
    schema_name: str,
    table_name: str,
) -> list[str]:
    """
    Return ordered primary-key columns.
    """
    cursor.execute(
        """
        SELECT
            attribute.attname
        FROM pg_constraint AS constraint_record
        INNER JOIN pg_class AS table_record
            ON table_record.oid =
               constraint_record.conrelid
        INNER JOIN pg_namespace AS namespace_record
            ON namespace_record.oid =
               table_record.relnamespace
        INNER JOIN unnest(
            constraint_record.conkey
        ) WITH ORDINALITY AS key_column(
            attribute_number,
            key_order
        )
            ON TRUE
        INNER JOIN pg_attribute AS attribute
            ON attribute.attrelid =
               table_record.oid
            AND attribute.attnum =
                key_column.attribute_number
        WHERE
            constraint_record.contype = 'p'
            AND namespace_record.nspname = %s
            AND table_record.relname = %s
        ORDER BY key_column.key_order;
        """,
        (
            schema_name,
            table_name,
        ),
    )

    return [
        row[0]
        for row in cursor.fetchall()
    ]


def fetch_constraint_names(
    cursor,
    schema_name: str,
    table_name: str,
) -> set[str]:
    """
    Return all named constraints attached to a table.
    """
    cursor.execute(
        """
        SELECT constraint_record.conname
        FROM pg_constraint AS constraint_record
        INNER JOIN pg_class AS table_record
            ON table_record.oid =
               constraint_record.conrelid
        INNER JOIN pg_namespace AS namespace_record
            ON namespace_record.oid =
               table_record.relnamespace
        WHERE
            namespace_record.nspname = %s
            AND table_record.relname = %s;
        """,
        (
            schema_name,
            table_name,
        ),
    )

    return {
        row[0]
        for row in cursor.fetchall()
    }


def fetch_index_names(
    cursor,
    schema_name: str,
    table_name: str,
) -> set[str]:
    """
    Return PostgreSQL index names for a table.
    """
    cursor.execute(
        """
        SELECT indexname
        FROM pg_indexes
        WHERE
            schemaname = %s
            AND tablename = %s;
        """,
        (
            schema_name,
            table_name,
        ),
    )

    return {
        row[0]
        for row in cursor.fetchall()
    }


def validate_table_contract(
    cursor,
    schema_name: str,
    table_name: str,
    expected_contract: dict,
) -> dict:
    """
    Validate a live table against its expected structural contract.
    """
    live_columns = fetch_table_columns(
        cursor=cursor,
        schema_name=schema_name,
        table_name=table_name,
    )

    expected_columns = (
        expected_contract["columns"]
    )

    missing_columns = sorted(
        set(expected_columns)
        - set(live_columns)
    )

    unexpected_columns = sorted(
        set(live_columns)
        - set(expected_columns)
    )

    type_mismatches = {}
    nullability_mismatches = {}

    for column_name in (
        set(expected_columns)
        & set(live_columns)
    ):
        expected_type = (
            normalise_type_spacing(
                expected_columns[
                    column_name
                ]["sql_type"]
            )
        )

        observed_type = (
            normalise_type_spacing(
                live_columns[
                    column_name
                ]["sql_type"]
            )
        )

        if expected_type != observed_type:
            type_mismatches[
                column_name
            ] = {
                "expected": expected_type,
                "observed": observed_type,
            }

        expected_nullable = (
            expected_columns[
                column_name
            ]["nullable"]
        )

        observed_nullable = (
            live_columns[
                column_name
            ]["nullable"]
        )

        if (
            expected_nullable
            != observed_nullable
        ):
            nullability_mismatches[
                column_name
            ] = {
                "expected": (
                    expected_nullable
                ),
                "observed": (
                    observed_nullable
                ),
            }

    observed_primary_key = (
        fetch_primary_key_columns(
            cursor=cursor,
            schema_name=schema_name,
            table_name=table_name,
        )
    )

    expected_primary_key = (
        expected_contract[
            "primary_key"
        ]
    )

    primary_key_matches = (
        observed_primary_key
        == expected_primary_key
    )

    observed_constraints = (
        fetch_constraint_names(
            cursor=cursor,
            schema_name=schema_name,
            table_name=table_name,
        )
    )

    missing_constraints = sorted(
        expected_contract[
            "required_constraints"
        ]
        - observed_constraints
    )

    observed_indexes = (
        fetch_index_names(
            cursor=cursor,
            schema_name=schema_name,
            table_name=table_name,
        )
    )

    missing_indexes = sorted(
        expected_contract[
            "required_indexes"
        ]
        - observed_indexes
    )

    contract_valid = not any(
        (
            missing_columns,
            unexpected_columns,
            type_mismatches,
            nullability_mismatches,
            not primary_key_matches,
            missing_constraints,
            missing_indexes,
        )
    )

    return {
        "schema_name": schema_name,
        "table_name": table_name,
        "valid": contract_valid,
        "missing_columns": missing_columns,
        "unexpected_columns": (
            unexpected_columns
        ),
        "type_mismatches": (
            type_mismatches
        ),
        "nullability_mismatches": (
            nullability_mismatches
        ),
        "expected_primary_key": (
            expected_primary_key
        ),
        "observed_primary_key": (
            observed_primary_key
        ),
        "missing_constraints": (
            missing_constraints
        ),
        "missing_indexes": (
            missing_indexes
        ),
    }


# ------------------------------------------
# Provision and validate tables
# ------------------------------------------

connection = None
cursor = None

tables_existing_before = []
tables_created_this_run = []
table_validation_results = []


log_database_event(
    message=(
        "DATABASE_TABLE_PROVISIONING_START: "
        f"schema_version="
        f"{DATABASE_ACTIVE_SCHEMA_VERSION}; "
        f"quality_rules_version="
        f"{DATABASE_QUALITY_RULESET_VERSION}."
    ),
    level="INFO",
)


try:
    connection = psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        sslmode=DB_SSLMODE,
        connect_timeout=10,
    )

    connection.autocommit = False
    cursor = connection.cursor()


    # --------------------------------------
    # Confirm required schemas still exist
    # --------------------------------------

    cursor.execute(
        """
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name = ANY(%s);
        """,
        (
            [
                "stg",
                "quarantine",
                "audit",
            ],
        ),
    )

    live_schema_names = {
        row[0]
        for row in cursor.fetchall()
    }

    missing_required_schemas = sorted(
        {
            "stg",
            "quarantine",
            "audit",
        }
        - live_schema_names
    )

    if missing_required_schemas:
        raise RuntimeError(
            "Required PostgreSQL schema(s) are missing: "
            + ", ".join(
                missing_required_schemas
            )
            + ". Rerun Part 4.1."
        )


    # --------------------------------------
    # Inspect tables before provisioning
    # --------------------------------------

    for schema_name, table_name in (
        EXPECTED_TABLE_CONTRACTS
    ):
        cursor.execute(
            """
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.tables
                WHERE
                    table_schema = %s
                    AND table_name = %s
                    AND table_type = 'BASE TABLE'
            );
            """,
            (
                schema_name,
                table_name,
            ),
        )

        if bool(cursor.fetchone()[0]):
            tables_existing_before.append(
                f"{schema_name}.{table_name}"
            )


    # --------------------------------------
    # Create audit.pipeline_job_log
    # --------------------------------------

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS
        audit.pipeline_job_log (
            job_id UUID NOT NULL,

            pipeline_session_id VARCHAR(64) NOT NULL,
            quality_audit_session_id VARCHAR(64) NOT NULL,

            source_file VARCHAR(255) NOT NULL,
            file_checksum VARCHAR(64) NOT NULL,

            status VARCHAR(30) NOT NULL,

            started_at TIMESTAMPTZ NOT NULL,
            completed_at TIMESTAMPTZ NULL,

            rows_read BIGINT NOT NULL DEFAULT 0,
            rows_loaded BIGINT NOT NULL DEFAULT 0,
            rows_quarantined BIGINT NOT NULL DEFAULT 0,

            duration_seconds NUMERIC(18, 3) NULL,

            schema_version VARCHAR(50) NOT NULL,
            quality_rules_version VARCHAR(50) NOT NULL,

            error_message TEXT NULL,

            created_at TIMESTAMPTZ NOT NULL
                DEFAULT CURRENT_TIMESTAMP,

            updated_at TIMESTAMPTZ NOT NULL
                DEFAULT CURRENT_TIMESTAMP,

            CONSTRAINT pk_pipeline_job_log
                PRIMARY KEY (job_id),

            CONSTRAINT ck_pipeline_job_log_status
                CHECK (
                    status IN (
                        'PENDING',
                        'RUNNING',
                        'SUCCESS',
                        'FAILED',
                        'SKIPPED'
                    )
                ),

            CONSTRAINT ck_pipeline_job_log_counts
                CHECK (
                    rows_read >= 0
                    AND rows_loaded >= 0
                    AND rows_quarantined >= 0
                )
        );
        """
    )


    # --------------------------------------
    # Create stg.cyclistic_rides
    # --------------------------------------

    cursor.execute(
        build_staging_table_statement()
    )


    # --------------------------------------
    # Create quarantine.cyclistic_rides
    # --------------------------------------

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS
        quarantine.cyclistic_rides (
            quarantine_id UUID NOT NULL,

            ride_id VARCHAR(255) NULL,

            failed_rule VARCHAR(100) NOT NULL,
            reason TEXT NOT NULL,

            source_file VARCHAR(255) NOT NULL,
            source_row_number BIGINT NOT NULL,

            raw_record JSONB NOT NULL,

            pipeline_session_id VARCHAR(64) NOT NULL,
            quality_audit_session_id VARCHAR(64) NOT NULL,

            created_at TIMESTAMPTZ NOT NULL
                DEFAULT CURRENT_TIMESTAMP,

            CONSTRAINT pk_quarantine_cyclistic_rides
                PRIMARY KEY (quarantine_id)
        );
        """
    )


    # --------------------------------------
    # Record newly created tables
    # --------------------------------------

    all_expected_table_names = {
        f"{schema_name}.{table_name}"
        for schema_name, table_name
        in EXPECTED_TABLE_CONTRACTS
    }

    tables_created_this_run = sorted(
        all_expected_table_names
        - set(tables_existing_before)
    )


    # --------------------------------------
    # Create audit-table indexes
    # --------------------------------------

    cursor.execute(
        """
        CREATE UNIQUE INDEX IF NOT EXISTS
        ux_pipeline_job_log_success_file
        ON audit.pipeline_job_log (
            source_file,
            file_checksum
        )
        WHERE status = 'SUCCESS';
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_pipeline_job_log_session
        ON audit.pipeline_job_log (
            pipeline_session_id
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_pipeline_job_log_source_file
        ON audit.pipeline_job_log (
            source_file
        );
        """
    )


    # --------------------------------------
    # Create staging indexes
    # --------------------------------------

    cursor.execute(
        """
        CREATE UNIQUE INDEX IF NOT EXISTS
        ux_stg_cyclistic_rides_ride_id
        ON stg.cyclistic_rides (
            ride_id
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_stg_cyclistic_rides_source
        ON stg.cyclistic_rides (
            source_file,
            source_row_number
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_stg_cyclistic_rides_session
        ON stg.cyclistic_rides (
            pipeline_session_id
        );
        """
    )


    # --------------------------------------
    # Create quarantine indexes
    # --------------------------------------

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_quarantine_cyclistic_rides_source
        ON quarantine.cyclistic_rides (
            source_file,
            source_row_number
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_quarantine_cyclistic_rides_rule
        ON quarantine.cyclistic_rides (
            failed_rule
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_quarantine_cyclistic_rides_session
        ON quarantine.cyclistic_rides (
            pipeline_session_id
        );
        """
    )


    # --------------------------------------
    # Validate all live table contracts
    # --------------------------------------

    for (
        schema_name,
        table_name,
    ), expected_contract in (
        EXPECTED_TABLE_CONTRACTS.items()
    ):
        validation_result = (
            validate_table_contract(
                cursor=cursor,
                schema_name=schema_name,
                table_name=table_name,
                expected_contract=(
                    expected_contract
                ),
            )
        )

        table_validation_results.append(
            validation_result
        )

        if not validation_result["valid"]:
            raise RuntimeError(
                "Database table contract mismatch for "
                f"{schema_name}.{table_name}: "
                f"{validation_result}"
            )

        log_database_event(
            message=(
                "DATABASE_TABLE_VERIFIED: "
                f"table={schema_name}.{table_name}; "
                f"columns="
                f"{len(expected_contract['columns'])}; "
                f"primary_key="
                f"{expected_contract['primary_key']}."
            ),
            level="INFO",
        )


    # --------------------------------------
    # Commit provisioning transaction
    # --------------------------------------

    connection.commit()

    log_database_event(
        message=(
            "DATABASE_TABLE_PROVISIONING_COMMITTED: "
            f"created="
            f"{len(tables_created_this_run)}; "
            f"existing="
            f"{len(tables_existing_before)}; "
            f"verified="
            f"{len(table_validation_results)}."
        ),
        level="INFO",
    )


except OperationalError as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_TABLE_PROVISIONING_FAILED: "
            "Unable to connect to PostgreSQL. "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise ConnectionError(
        "Unable to open the PostgreSQL connection required "
        "for Part 4.2."
    ) from error


except (
    PermissionError,
    RuntimeError,
    ValueError,
    TypeError,
    psycopg2.DatabaseError,
) as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_TABLE_PROVISIONING_ROLLED_BACK: "
            f"error_type={type(error).__name__}; "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise


finally:
    if cursor is not None:
        cursor.close()
        print("✓ Database cursor closed.")

    if connection is not None:
        connection.close()
        print("✓ Database connection closed.")


# ------------------------------------------
# Build Part 4.2 summary
# ------------------------------------------

DATABASE_TABLE_PROVISIONING_SUMMARY = {
    "database_processing_session_id": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "schema_version": (
        DATABASE_ACTIVE_SCHEMA_VERSION
    ),
    "quality_rules_version": (
        DATABASE_QUALITY_RULESET_VERSION
    ),
    "tables_existing_before": deepcopy(
        tables_existing_before
    ),
    "tables_created_this_run": deepcopy(
        tables_created_this_run
    ),
    "table_validation_results": deepcopy(
        table_validation_results
    ),
    "status": "SUCCESS",
    "completed_at": datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds"),
}


# ------------------------------------------
# Display Part 4.2 summary
# ------------------------------------------

print(
    "\n========== PostgreSQL Table Provisioning "
    "==========\n"
)

print(
    f"Database-processing session : "
    f"{DATABASE_PROCESSING_SESSION_ID}"
)
print(
    f"Approved schema version     : "
    f"{DATABASE_ACTIVE_SCHEMA_VERSION}"
)
print(
    f"Quality ruleset version     : "
    f"{DATABASE_QUALITY_RULESET_VERSION}"
)
print(
    f"Tables created              : "
    f"{len(tables_created_this_run)}"
)
print(
    f"Tables already existing     : "
    f"{len(tables_existing_before)}"
)
print(
    f"Tables verified             : "
    f"{len(table_validation_results)}"
)


print("\nTable details:")

for validation_result in (
    table_validation_results
):
    qualified_table_name = (
        f"{validation_result['schema_name']}."
        f"{validation_result['table_name']}"
    )

    print(
        f"  {qualified_table_name:<38} "
        f"columns="
        f"{len(EXPECTED_TABLE_CONTRACTS[
            (
                validation_result['schema_name'],
                validation_result['table_name'],
            )
        ]['columns']):<3} "
        f"status=VALID"
    )


print("\n" + "=" * 78)
print("✓ Required PostgreSQL tables exist.")
print("✓ Approved schema applied to the staging table.")
print("✓ Lineage columns provisioned.")
print("✓ Quarantine JSONB structure provisioned.")
print("✓ Job-log idempotency index provisioned.")
print("✓ Columns, keys, constraints, and indexes validated.")
print("✓ Part 4.2 completed successfully.")
print("✓ Ready for Part 4.3 load-session initialisation.")
print("=" * 78)

## Part 4.3: Initialise File Load Plans and Check Idempotency

### Objective

Prepare each discovered source CSV for database loading without inserting any source records yet.

This section calculates a stable identity for every source file, checks the database job history, and determines whether the file should be loaded or skipped.

### Quality-Audit Dependency

Part 4 must use the completed quality-audit run produced in Part 3.

Before creating the load plan, this section reloads:

```text
logs/data_quality/latest_quality_audit.json
```

It verifies that:

* the quality audit execution status is `COMPLETE`;
* the quality-audit session ID is available;
* the audited schema version matches the approved database schema version;
* the quality-ruleset version matches the version used to provision the database tables.

A quality outcome of `FAIL` does not block database processing automatically. It means failure-severity rows must be routed to quarantine during Part 4.4.

### File Identity

Each source file is identified using:

```text
source filename + SHA-256 checksum
```

The checksum is calculated from the complete file contents in binary chunks.

Filename alone is insufficient because a file may be replaced while retaining the same name.

### Idempotency Policy

For each discovered file, the pipeline queries:

```text
audit.pipeline_job_log
```

If the same filename and checksum already has:

```text
status = SUCCESS
```

the file is assigned:

```text
SKIPPED_ALREADY_LOADED
```

Otherwise, it is assigned:

```text
READY
```

and receives a new `PENDING` job record.

### Rerun Safety

If Part 4.3 is rerun within the same database-processing session, it reuses an existing matching `PENDING` or `SKIPPED` job record rather than creating duplicates.

A new database-processing session may create a new skipped-job record to preserve execution history.

### Job Records

Ready files receive a `PENDING` record containing:

* job ID;
* pipeline session ID;
* quality-audit session ID;
* source filename;
* SHA-256 checksum;
* approved schema version;
* quality-ruleset version;
* initial row counts of zero;
* start timestamp.

Skipped files receive a completed `SKIPPED` record unless the same skipped record already exists for the current pipeline session.

### Transaction Policy

All job-plan records are inserted in one transaction.

If one job record cannot be created:

```text
Rollback all new Part 4.3 job records
```

No source data is loaded during this section.

### In-Memory Outputs

This section produces:

```text
DATABASE_FILE_LOAD_PLAN
DATABASE_READY_FILES
DATABASE_SKIPPED_FILES
DATABASE_LOAD_PLAN_SUMMARY
```

Part 4.4 will use `DATABASE_READY_FILES` to perform file-by-file loading and routing.

### Expected First Execution

```text
Files discovered : 4
Files ready       : 4
Files skipped     : 0
Pending jobs      : 4
```

### Expected Later Execution

When all files have already been loaded successfully and their contents have not changed:

```text
Files discovered : 4
Files ready       : 0
Files skipped     : 4
Pending jobs      : 0
```


In [ ]:
# ==========================================
# Part 4.3: Initialise File Load Plans and
# Check Idempotency
# ==========================================

import hashlib
import json
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import psycopg2
from psycopg2 import OperationalError


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_4_3_VALUES = {
    "DATABASE_PROCESSING_SESSION_ID": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "DATABASE_ACTIVE_SCHEMA_VERSION": (
        DATABASE_ACTIVE_SCHEMA_VERSION
    ),
    "DATABASE_QUALITY_RULESET_VERSION": (
        DATABASE_QUALITY_RULESET_VERSION
    ),
    "DB_HOST": DB_HOST,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_PORT": DB_PORT,
    "DB_SSLMODE": DB_SSLMODE,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
    "discovered_files": discovered_files,
}

missing_part_4_3_values = [
    variable_name
    for variable_name, variable_value
    in REQUIRED_PART_4_3_VALUES.items()
    if variable_value is None
]

if missing_part_4_3_values:
    raise RuntimeError(
        "Part 4.3 is missing required value(s): "
        + ", ".join(
            sorted(missing_part_4_3_values)
        )
    )


if not isinstance(LOG_AUDIT_PATH, Path):
    raise TypeError(
        "LOG_AUDIT_PATH must be a pathlib.Path object."
    )


if not isinstance(discovered_files, list):
    raise TypeError(
        "discovered_files must be a list."
    )


if not discovered_files:
    raise RuntimeError(
        "Part 4.3 cannot continue because no source files "
        "were discovered."
    )


for file_path in discovered_files:
    if not isinstance(file_path, Path):
        raise TypeError(
            "Every discovered file must be a pathlib.Path object."
        )

    if not file_path.exists():
        raise FileNotFoundError(
            f"Discovered source file does not exist:\n{file_path}"
        )

    if not file_path.is_file():
        raise FileExistsError(
            f"Discovered source path is not a file:\n{file_path}"
        )


# ------------------------------------------
# Load latest completed quality-audit pointer
# ------------------------------------------

LATEST_QUALITY_AUDIT_POINTER_PATH = (
    LOG_AUDIT_PATH
    / "data_quality"
    / "latest_quality_audit.json"
)


if not LATEST_QUALITY_AUDIT_POINTER_PATH.exists():
    raise FileNotFoundError(
        "Latest quality-audit pointer does not exist. "
        "Complete Part 3.4 before running Part 4.3:\n"
        f"{LATEST_QUALITY_AUDIT_POINTER_PATH}"
    )


try:
    with LATEST_QUALITY_AUDIT_POINTER_PATH.open(
        mode="r",
        encoding="utf-8",
    ) as latest_quality_file:
        latest_quality_audit = json.load(
            latest_quality_file
        )

except json.JSONDecodeError as error:
    raise ValueError(
        "latest_quality_audit.json contains invalid JSON."
    ) from error


if not isinstance(latest_quality_audit, dict):
    raise TypeError(
        "latest_quality_audit.json must contain a JSON object."
    )


LOAD_QUALITY_AUDIT_SESSION_ID = (
    latest_quality_audit.get(
        "quality_audit_session_id"
    )
)

LOAD_QUALITY_EXECUTION_STATUS = (
    latest_quality_audit.get(
        "execution_status"
    )
)

LOAD_QUALITY_OUTCOME = (
    latest_quality_audit.get(
        "overall_quality_outcome"
    )
)

LOAD_QUALITY_SCHEMA_VERSION = (
    latest_quality_audit.get(
        "source_schema_version"
    )
)

LOAD_QUALITY_RULESET_VERSION = (
    latest_quality_audit.get(
        "ruleset_version"
    )
)


if LOAD_QUALITY_EXECUTION_STATUS != "COMPLETE":
    raise RuntimeError(
        "Part 4 requires a completed quality audit. "
        f"Observed execution_status="
        f"{LOAD_QUALITY_EXECUTION_STATUS}."
    )


if (
    not isinstance(
        LOAD_QUALITY_AUDIT_SESSION_ID,
        str,
    )
    or not LOAD_QUALITY_AUDIT_SESSION_ID.strip()
):
    raise RuntimeError(
        "Latest quality audit has no valid session ID."
    )


if (
    LOAD_QUALITY_SCHEMA_VERSION
    != DATABASE_ACTIVE_SCHEMA_VERSION
):
    raise RuntimeError(
        "Quality-audit schema version does not match the "
        "database table contract. "
        f"Quality audit={LOAD_QUALITY_SCHEMA_VERSION}; "
        f"Database={DATABASE_ACTIVE_SCHEMA_VERSION}."
    )


if (
    LOAD_QUALITY_RULESET_VERSION
    != DATABASE_QUALITY_RULESET_VERSION
):
    raise RuntimeError(
        "Quality ruleset version does not match the "
        "database table contract. "
        f"Quality audit={LOAD_QUALITY_RULESET_VERSION}; "
        f"Database={DATABASE_QUALITY_RULESET_VERSION}."
    )


# ------------------------------------------
# Checksum configuration
# ------------------------------------------

FILE_CHECKSUM_CHUNK_SIZE = int(
    config.get(
        "FILE_CHECKSUM_CHUNK_SIZE",
        1024 * 1024,
    )
)


if FILE_CHECKSUM_CHUNK_SIZE <= 0:
    raise ValueError(
        "FILE_CHECKSUM_CHUNK_SIZE must be greater than zero."
    )


def calculate_sha256(
    file_path: Path,
) -> str:
    """
    Calculate a complete SHA-256 checksum without loading the
    entire file into memory.
    """
    checksum = hashlib.sha256()

    with file_path.open(
        mode="rb",
    ) as source_file:
        while True:
            binary_chunk = source_file.read(
                FILE_CHECKSUM_CHUNK_SIZE
            )

            if not binary_chunk:
                break

            checksum.update(binary_chunk)

    return checksum.hexdigest()


# ------------------------------------------
# Initialise in-memory load-plan outputs
# ------------------------------------------

DATABASE_FILE_LOAD_PLAN = []
DATABASE_READY_FILES = []
DATABASE_SKIPPED_FILES = []

file_identity_records = []


# ------------------------------------------
# Calculate file identities
# ------------------------------------------

log_database_event(
    message=(
        "DATABASE_LOAD_PLAN_START: "
        f"files={len(discovered_files)}; "
        f"quality_audit_session="
        f"{LOAD_QUALITY_AUDIT_SESSION_ID}; "
        f"quality_outcome={LOAD_QUALITY_OUTCOME}."
    ),
    level="INFO",
)


for file_path in sorted(
    discovered_files,
    key=lambda path: path.name,
):
    file_checksum = calculate_sha256(
        file_path
    )

    file_identity = {
        "source_file": file_path.name,
        "source_path": file_path,
        "file_checksum": file_checksum,
        "file_size_bytes": (
            file_path.stat().st_size
        ),
    }

    file_identity_records.append(
        file_identity
    )

    log_database_event(
        message=(
            "DATABASE_FILE_CHECKSUM_COMPLETE: "
            f"file={file_path.name}; "
            f"sha256={file_checksum}; "
            f"size_bytes="
            f"{file_identity['file_size_bytes']}."
        ),
        level="INFO",
    )


# ------------------------------------------
# Create load plan and job records
# ------------------------------------------

connection = None
cursor = None

pending_jobs_created = 0
pending_jobs_reused = 0
skipped_jobs_created = 0
skipped_jobs_reused = 0


try:
    connection = psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        sslmode=DB_SSLMODE,
        connect_timeout=10,
    )

    connection.autocommit = False
    cursor = connection.cursor()


    # --------------------------------------
    # Confirm required job table exists
    # --------------------------------------

    cursor.execute(
        """
        SELECT EXISTS (
            SELECT 1
            FROM information_schema.tables
            WHERE
                table_schema = 'audit'
                AND table_name = 'pipeline_job_log'
                AND table_type = 'BASE TABLE'
        );
        """
    )

    if not bool(cursor.fetchone()[0]):
        raise RuntimeError(
            "audit.pipeline_job_log does not exist. "
            "Rerun Part 4.2."
        )


    for file_identity in file_identity_records:

        source_file = file_identity[
            "source_file"
        ]

        file_checksum = file_identity[
            "file_checksum"
        ]

        source_path = file_identity[
            "source_path"
        ]


        # ----------------------------------
        # Check global successful history
        # ----------------------------------

        cursor.execute(
            """
            SELECT
                job_id,
                pipeline_session_id,
                completed_at
            FROM audit.pipeline_job_log
            WHERE
                source_file = %s
                AND file_checksum = %s
                AND status = 'SUCCESS'
            ORDER BY completed_at DESC
            LIMIT 1;
            """,
            (
                source_file,
                file_checksum,
            ),
        )

        successful_job = cursor.fetchone()


        if successful_job is not None:
            (
                successful_job_id,
                successful_pipeline_session,
                successful_completed_at,
            ) = successful_job


            # Check whether this session already recorded the skip.
            cursor.execute(
                """
                SELECT job_id
                FROM audit.pipeline_job_log
                WHERE
                    pipeline_session_id = %s
                    AND source_file = %s
                    AND file_checksum = %s
                    AND status = 'SKIPPED'
                ORDER BY created_at DESC
                LIMIT 1;
                """,
                (
                    DATABASE_PROCESSING_SESSION_ID,
                    source_file,
                    file_checksum,
                ),
            )

            current_skip_record = (
                cursor.fetchone()
            )

            if current_skip_record is not None:
                skipped_job_id = (
                    current_skip_record[0]
                )

                skipped_jobs_reused += 1

            else:
                skipped_job_id = uuid4()

                current_timestamp = (
                    datetime.now(
                        MELBOURNE_TIMEZONE
                    )
                )

                cursor.execute(
                    """
                    INSERT INTO audit.pipeline_job_log (
                        job_id,
                        pipeline_session_id,
                        quality_audit_session_id,
                        source_file,
                        file_checksum,
                        status,
                        started_at,
                        completed_at,
                        rows_read,
                        rows_loaded,
                        rows_quarantined,
                        duration_seconds,
                        schema_version,
                        quality_rules_version,
                        error_message,
                        created_at,
                        updated_at
                    )
                    VALUES (
                        %s, %s, %s, %s, %s,
                        'SKIPPED',
                        %s, %s,
                        0, 0, 0,
                        0,
                        %s, %s,
                        %s,
                        %s, %s
                    );
                    """,
                    (
                        str(skipped_job_id),
                        DATABASE_PROCESSING_SESSION_ID,
                        LOAD_QUALITY_AUDIT_SESSION_ID,
                        source_file,
                        file_checksum,
                        current_timestamp,
                        current_timestamp,
                        DATABASE_ACTIVE_SCHEMA_VERSION,
                        DATABASE_QUALITY_RULESET_VERSION,
                        (
                            "Already loaded successfully by "
                            f"job {successful_job_id} in pipeline "
                            f"session {successful_pipeline_session}."
                        ),
                        current_timestamp,
                        current_timestamp,
                    ),
                )

                skipped_jobs_created += 1


            skipped_plan_record = {
                **file_identity,
                "job_id": str(
                    skipped_job_id
                ),
                "load_decision": (
                    "SKIPPED_ALREADY_LOADED"
                ),
                "job_status": "SKIPPED",
                "previous_successful_job_id": str(
                    successful_job_id
                ),
                "previous_pipeline_session_id": (
                    successful_pipeline_session
                ),
                "previous_completed_at": (
                    successful_completed_at
                ),
                "quality_audit_session_id": (
                    LOAD_QUALITY_AUDIT_SESSION_ID
                ),
            }

            DATABASE_FILE_LOAD_PLAN.append(
                skipped_plan_record
            )

            DATABASE_SKIPPED_FILES.append(
                skipped_plan_record
            )

            log_database_event(
                message=(
                    "DATABASE_FILE_SKIPPED: "
                    f"file={source_file}; "
                    "reason=ALREADY_LOADED_SUCCESSFULLY; "
                    f"previous_job={successful_job_id}."
                ),
                level="INFO",
            )

            continue


        # ----------------------------------
        # Reuse an existing current-session
        # PENDING job when Part 4.3 is rerun
        # ----------------------------------

        cursor.execute(
            """
            SELECT job_id
            FROM audit.pipeline_job_log
            WHERE
                pipeline_session_id = %s
                AND source_file = %s
                AND file_checksum = %s
                AND status = 'PENDING'
            ORDER BY created_at DESC
            LIMIT 1;
            """,
            (
                DATABASE_PROCESSING_SESSION_ID,
                source_file,
                file_checksum,
            ),
        )

        current_pending_job = (
            cursor.fetchone()
        )


        if current_pending_job is not None:
            pending_job_id = (
                current_pending_job[0]
            )

            pending_jobs_reused += 1

        else:
            pending_job_id = uuid4()

            current_timestamp = (
                datetime.now(
                    MELBOURNE_TIMEZONE
                )
            )

            cursor.execute(
                """
                INSERT INTO audit.pipeline_job_log (
                    job_id,
                    pipeline_session_id,
                    quality_audit_session_id,
                    source_file,
                    file_checksum,
                    status,
                    started_at,
                    completed_at,
                    rows_read,
                    rows_loaded,
                    rows_quarantined,
                    duration_seconds,
                    schema_version,
                    quality_rules_version,
                    error_message,
                    created_at,
                    updated_at
                )
                VALUES (
                    %s, %s, %s, %s, %s,
                    'PENDING',
                    %s, NULL,
                    0, 0, 0,
                    NULL,
                    %s, %s,
                    NULL,
                    %s, %s
                );
                """,
                (
                    str(pending_job_id),
                    DATABASE_PROCESSING_SESSION_ID,
                    LOAD_QUALITY_AUDIT_SESSION_ID,
                    source_file,
                    file_checksum,
                    current_timestamp,
                    DATABASE_ACTIVE_SCHEMA_VERSION,
                    DATABASE_QUALITY_RULESET_VERSION,
                    current_timestamp,
                    current_timestamp,
                ),
            )

            pending_jobs_created += 1


        ready_plan_record = {
            **file_identity,
            "job_id": str(
                pending_job_id
            ),
            "load_decision": "READY",
            "job_status": "PENDING",
            "quality_audit_session_id": (
                LOAD_QUALITY_AUDIT_SESSION_ID
            ),
        }

        DATABASE_FILE_LOAD_PLAN.append(
            ready_plan_record
        )

        DATABASE_READY_FILES.append(
            ready_plan_record
        )

        log_database_event(
            message=(
                "DATABASE_FILE_READY: "
                f"file={source_file}; "
                f"job_id={pending_job_id}; "
                f"checksum={file_checksum}."
            ),
            level="INFO",
        )


    connection.commit()

    log_database_event(
        message=(
            "DATABASE_LOAD_PLAN_COMMITTED: "
            f"ready={len(DATABASE_READY_FILES)}; "
            f"skipped={len(DATABASE_SKIPPED_FILES)}; "
            f"pending_created={pending_jobs_created}; "
            f"pending_reused={pending_jobs_reused}; "
            f"skipped_created={skipped_jobs_created}; "
            f"skipped_reused={skipped_jobs_reused}."
        ),
        level="INFO",
    )


except OperationalError as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_LOAD_PLAN_FAILED: "
            "Unable to connect to PostgreSQL. "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise ConnectionError(
        "Unable to open the PostgreSQL connection required "
        "for Part 4.3."
    ) from error


except (
    RuntimeError,
    ValueError,
    TypeError,
    OSError,
    psycopg2.DatabaseError,
) as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_LOAD_PLAN_ROLLED_BACK: "
            f"error_type={type(error).__name__}; "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise


finally:
    if cursor is not None:
        cursor.close()
        print("✓ Database cursor closed.")

    if connection is not None:
        connection.close()
        print("✓ Database connection closed.")


# ------------------------------------------
# Build Part 4.3 summary
# ------------------------------------------

DATABASE_LOAD_PLAN_SUMMARY = {
    "database_processing_session_id": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "quality_audit_session_id": (
        LOAD_QUALITY_AUDIT_SESSION_ID
    ),
    "quality_audit_execution_status": (
        LOAD_QUALITY_EXECUTION_STATUS
    ),
    "quality_outcome": (
        LOAD_QUALITY_OUTCOME
    ),
    "schema_version": (
        DATABASE_ACTIVE_SCHEMA_VERSION
    ),
    "quality_rules_version": (
        DATABASE_QUALITY_RULESET_VERSION
    ),
    "files_discovered": len(
        DATABASE_FILE_LOAD_PLAN
    ),
    "files_ready": len(
        DATABASE_READY_FILES
    ),
    "files_skipped": len(
        DATABASE_SKIPPED_FILES
    ),
    "pending_jobs_created": (
        pending_jobs_created
    ),
    "pending_jobs_reused": (
        pending_jobs_reused
    ),
    "skipped_jobs_created": (
        skipped_jobs_created
    ),
    "skipped_jobs_reused": (
        skipped_jobs_reused
    ),
    "status": "SUCCESS",
    "completed_at": datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds"),
}


# ------------------------------------------
# Display Part 4.3 summary
# ------------------------------------------

print(
    "\n========== Database File Load Plan "
    "==========\n"
)

print(
    f"Database-processing session : "
    f"{DATABASE_PROCESSING_SESSION_ID}"
)
print(
    f"Quality-audit session       : "
    f"{LOAD_QUALITY_AUDIT_SESSION_ID}"
)
print(
    f"Quality-audit execution     : "
    f"{LOAD_QUALITY_EXECUTION_STATUS}"
)
print(
    f"Quality outcome             : "
    f"{LOAD_QUALITY_OUTCOME}"
)
print(
    f"Files discovered            : "
    f"{len(DATABASE_FILE_LOAD_PLAN)}"
)
print(
    f"Files ready                 : "
    f"{len(DATABASE_READY_FILES)}"
)
print(
    f"Files skipped               : "
    f"{len(DATABASE_SKIPPED_FILES)}"
)
print(
    f"Pending jobs created        : "
    f"{pending_jobs_created}"
)
print(
    f"Pending jobs reused         : "
    f"{pending_jobs_reused}"
)


print("\nFile decisions:")

for plan_record in DATABASE_FILE_LOAD_PLAN:
    print(
        f"  {plan_record['source_file']:<40} "
        f"{plan_record['load_decision']:<28} "
        f"job={plan_record['job_id']}"
    )


print("\n" + "=" * 78)

if DATABASE_READY_FILES:
    print(
        "✓ Eligible files have PENDING database job records."
    )
else:
    print(
        "✓ No files require loading; all matching files were skipped."
    )

print("✓ File checksums calculated using SHA-256.")
print("✓ Idempotency history checked.")
print("✓ Load plan transaction committed.")
print("✓ Part 4.3 completed successfully.")
print("✓ Ready for Part 4.4 file loading and quarantine routing.")
print("=" * 78)

## Part 4.4: Load and Route Source Files

### Objective

Load each eligible source CSV into PostgreSQL while routing failure-severity records to the quarantine table.

Part 4.3 created one `PENDING` job for each eligible file. Part 4.4 processes those files individually using one database transaction per source file.

### Processing Model

For each file:

```text
Verify file checksum
↓
Set job status to RUNNING
↓
Read source file in chunks
↓
Re-evaluate failure rules at row level
↓
Valid rows → stg.cyclistic_rides
Failed rows → quarantine.cyclistic_rides
↓
Reconcile counts
↓
Set job status to SUCCESS
↓
Commit file transaction
```

If processing fails:

```text
Rollback the current file transaction
↓
Preserve previously committed files
↓
Update the file job to FAILED in a separate recovery transaction
```

### Row-Level Rule Evaluation

Part 3 retained complete metric counts but only bounded evidence. Part 4.4 must therefore re-evaluate failure rules against every source row.

Supported failure-rule types are:

* `NOT_NULL`
* `NOT_BLANK`
* `ALLOWED_VALUES`
* `DATETIME_PARSEABLE`
* `DATETIME_ORDER`
* `NUMERIC_RANGE`

The dataset-level uniqueness rule is not recalculated during insertion. Part 4.4 requires the completed Part 3 audit to report zero duplicate `ride_id` values before loading begins. The unique database index on `stg.cyclistic_rides.ride_id` remains a final defensive control.

### Routing Policy

A source row is loaded into staging only when it violates no enabled failure-severity rule.

A failed source row is excluded from staging and receives one quarantine record for each violated failure rule.

For example, one row that violates two failure rules produces:

```text
0 staging rows
2 quarantine findings
```

The quarantine table preserves:

* source filename;
* source row number;
* ride ID;
* failed rule ID;
* failure reason;
* complete source record as `JSONB`;
* pipeline and quality-audit session IDs.

### Warning Rules

Warning-only conditions do not block loading.

Examples include:

* missing station fields;
* missing coordinates;
* rides lasting more than 24 hours.

These records remain eligible for staging because their warnings were already recorded by Part 3.

### Chunked Loading

Source files are read in configurable chunks.

The default settings are:

```text
DB_LOAD_CHUNK_SIZE = 50,000 rows
DB_INSERT_PAGE_SIZE = 10,000 rows
```

Only the current chunk and its prepared insert records are retained in memory.

### File Identity Protection

Before loading each file, its SHA-256 checksum is recalculated and compared with the checksum stored in the Part 4.3 load plan.

If the file changed after Part 4.3:

```text
Abort the file
Mark the job FAILED
Do not load modified contents under the old job identity
```

### Reconciliation

Before committing each file:

```text
rows read
=
rows loaded
+
unique source rows quarantined
```

The number of quarantine table records may be greater than the number of quarantined source rows because one row may violate multiple failure rules.

### Expected Outcome for the Current Dataset

```text
202509 source rows: 714,759
202510 source rows: 646,039
202511 source rows: 356,628
202512 source rows: 140,534
```

Expected routing:

```text
Staging source rows    : 1,857,931
Quarantined source rows: 29
```

The 29 quarantined rows are expected to violate:

```text
DQ007_END_AFTER_START
```


In [ ]:
# ==========================================
# Part 4.4: Load and Route Source Files
# ==========================================

import hashlib
import json
import math
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import pandas as pd
import psycopg2
from psycopg2 import OperationalError, sql
from psycopg2.extras import Json, execute_values


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_4_4_VALUES = {
    "DATABASE_PROCESSING_SESSION_ID": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "DATABASE_READY_FILES": (
        DATABASE_READY_FILES
    ),
    "DATABASE_ACTIVE_SCHEMA_VERSION": (
        DATABASE_ACTIVE_SCHEMA_VERSION
    ),
    "DATABASE_QUALITY_RULESET_VERSION": (
        DATABASE_QUALITY_RULESET_VERSION
    ),
    "NORMALISED_ACTIVE_SCHEMA": (
        NORMALISED_ACTIVE_SCHEMA
    ),
    "LOAD_QUALITY_AUDIT_SESSION_ID": (
        LOAD_QUALITY_AUDIT_SESSION_ID
    ),
    "LATEST_QUALITY_AUDIT_POINTER_PATH": (
        LATEST_QUALITY_AUDIT_POINTER_PATH
    ),
    "DB_HOST": DB_HOST,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_PORT": DB_PORT,
    "DB_SSLMODE": DB_SSLMODE,
    "MELBOURNE_TIMEZONE": (
        MELBOURNE_TIMEZONE
    ),
}

missing_part_4_4_values = [
    variable_name
    for variable_name, variable_value
    in REQUIRED_PART_4_4_VALUES.items()
    if variable_value is None
]

if missing_part_4_4_values:
    raise RuntimeError(
        "Part 4.4 is missing required value(s): "
        + ", ".join(
            sorted(missing_part_4_4_values)
        )
    )


if not isinstance(
    DATABASE_READY_FILES,
    list,
):
    raise TypeError(
        "DATABASE_READY_FILES must be a list."
    )


if not isinstance(
    NORMALISED_ACTIVE_SCHEMA,
    dict,
) or not NORMALISED_ACTIVE_SCHEMA:
    raise RuntimeError(
        "Part 4.4 requires the approved database schema."
    )


# ------------------------------------------
# Load configuration
# ------------------------------------------

DB_LOAD_CHUNK_SIZE = int(
    config.get(
        "DB_LOAD_CHUNK_SIZE",
        50_000,
    )
)

DB_INSERT_PAGE_SIZE = int(
    config.get(
        "DB_INSERT_PAGE_SIZE",
        10_000,
    )
)

FILE_CHECKSUM_CHUNK_SIZE = int(
    config.get(
        "FILE_CHECKSUM_CHUNK_SIZE",
        1024 * 1024,
    )
)


for setting_name, setting_value in {
    "DB_LOAD_CHUNK_SIZE": (
        DB_LOAD_CHUNK_SIZE
    ),
    "DB_INSERT_PAGE_SIZE": (
        DB_INSERT_PAGE_SIZE
    ),
    "FILE_CHECKSUM_CHUNK_SIZE": (
        FILE_CHECKSUM_CHUNK_SIZE
    ),
}.items():
    if setting_value <= 0:
        raise ValueError(
            f"{setting_name} must be greater than zero."
        )


# ------------------------------------------
# Reload latest quality-audit artifacts
# ------------------------------------------

with LATEST_QUALITY_AUDIT_POINTER_PATH.open(
    mode="r",
    encoding="utf-8",
) as latest_quality_file:
    latest_quality_pointer = json.load(
        latest_quality_file
    )


quality_bundle_path = Path(
    latest_quality_pointer[
        "artifacts"
    ][
        "audit_bundle"
    ]
)


if not quality_bundle_path.exists():
    raise FileNotFoundError(
        "Quality-audit bundle does not exist:\n"
        f"{quality_bundle_path}"
    )


with quality_bundle_path.open(
    mode="r",
    encoding="utf-8",
) as quality_bundle_file:
    quality_audit_bundle = json.load(
        quality_bundle_file
    )


if (
    quality_audit_bundle[
        "audit_metadata"
    ][
        "quality_audit_session_id"
    ]
    != LOAD_QUALITY_AUDIT_SESSION_ID
):
    raise RuntimeError(
        "Quality-audit bundle session does not match "
        "the Part 4 load plan."
    )


audited_duplicate_count = int(
    quality_audit_bundle[
        "dataset_metrics"
    ][
        "duplicates"
    ][
        "duplicate_ride_id_count"
    ]
)


if audited_duplicate_count != 0:
    raise RuntimeError(
        "Part 4.4 currently requires zero duplicate ride IDs "
        "in the completed quality audit. "
        f"Observed duplicates={audited_duplicate_count}."
    )


# ------------------------------------------
# Reload enabled failure rules
# ------------------------------------------

with QUALITY_RULES_PATH.open(
    mode="r",
    encoding="utf-8",
) as quality_rules_file:
    database_load_quality_rules = json.load(
        quality_rules_file
    )


ENABLED_FAILURE_RULES = {
    rule_id: rule_definition
    for rule_id, rule_definition in (
        database_load_quality_rules[
            "rules"
        ].items()
    )
    if (
        rule_definition.get(
            "enabled"
        ) is True
        and rule_definition.get(
            "severity"
        ) == "FAIL"
    )
}


SUPPORTED_DATABASE_FAILURE_RULE_TYPES = {
    "NOT_NULL",
    "NOT_BLANK",
    "ALLOWED_VALUES",
    "DATETIME_PARSEABLE",
    "DATETIME_ORDER",
    "NUMERIC_RANGE",
    "UNIQUE",
}


unsupported_failure_rules = {
    rule_id: rule_definition.get(
        "rule_type"
    )
    for rule_id, rule_definition
    in ENABLED_FAILURE_RULES.items()
    if rule_definition.get(
        "rule_type"
    )
    not in SUPPORTED_DATABASE_FAILURE_RULE_TYPES
}


if unsupported_failure_rules:
    raise RuntimeError(
        "Part 4.4 contains unsupported failure rules: "
        f"{unsupported_failure_rules}"
    )


# Dataset uniqueness was already established by Part 3.
ROW_LEVEL_FAILURE_RULES = {
    rule_id: rule_definition
    for rule_id, rule_definition
    in ENABLED_FAILURE_RULES.items()
    if rule_definition[
        "rule_type"
    ] != "UNIQUE"
}


# ------------------------------------------
# Define source-column handling
# ------------------------------------------

APPROVED_SOURCE_COLUMNS = list(
    NORMALISED_ACTIVE_SCHEMA.keys()
)


TEXT_SQL_PREFIXES = (
    "VARCHAR",
    "TEXT",
)


def is_text_sql_type(
    declared_type: str,
) -> bool:
    return declared_type.startswith(
        TEXT_SQL_PREFIXES
    )


TEXT_SOURCE_COLUMNS = {
    column_name
    for column_name, declared_type
    in NORMALISED_ACTIVE_SCHEMA.items()
    if is_text_sql_type(declared_type)
}


DATETIME_SOURCE_COLUMNS = {
    column_name
    for column_name, declared_type
    in NORMALISED_ACTIVE_SCHEMA.items()
    if declared_type.startswith(
        "TIMESTAMP"
    )
}


NUMERIC_SOURCE_COLUMNS = {
    column_name
    for column_name, declared_type
    in NORMALISED_ACTIVE_SCHEMA.items()
    if declared_type.startswith(
        (
            "NUMERIC",
            "INTEGER",
            "BIGINT",
            "SMALLINT",
            "REAL",
            "DOUBLE PRECISION",
        )
    )
}


CSV_EXPLICIT_DTYPES = {
    column_name: "string"
    for column_name
    in TEXT_SOURCE_COLUMNS
}


# ------------------------------------------
# General helpers
# ------------------------------------------

def calculate_file_sha256(
    file_path: Path,
) -> str:
    """
    Calculate a full SHA-256 checksum using bounded memory.
    """
    checksum = hashlib.sha256()

    with file_path.open(
        mode="rb",
    ) as source_file:
        while True:
            binary_chunk = source_file.read(
                FILE_CHECKSUM_CHUNK_SIZE
            )

            if not binary_chunk:
                break

            checksum.update(binary_chunk)

    return checksum.hexdigest()


def python_safe_value(
    value,
):
    """
    Convert Pandas and NumPy scalar values into psycopg2-safe
    Python values.
    """
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (
        TypeError,
        ValueError,
    ):
        pass

    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()

    if hasattr(value, "item"):
        try:
            return value.item()
        except (
            ValueError,
            TypeError,
            AttributeError,
        ):
            pass

    return value


def json_safe_record(
    row: pd.Series,
) -> dict:
    """
    Convert one source row into a JSON-compatible dictionary.
    """
    safe_record = {}

    for column_name, value in row.items():

        if value is None:
            safe_record[column_name] = None
            continue

        try:
            if pd.isna(value):
                safe_record[column_name] = None
                continue
        except (
            TypeError,
            ValueError,
        ):
            pass

        if isinstance(value, pd.Timestamp):
            safe_record[
                column_name
            ] = value.isoformat()

        elif hasattr(value, "item"):
            try:
                safe_record[
                    column_name
                ] = value.item()
            except (
                ValueError,
                TypeError,
                AttributeError,
            ):
                safe_record[
                    column_name
                ] = str(value)

        else:
            safe_record[
                column_name
            ] = value

    return safe_record


def prepare_chunk_types(
    raw_chunk: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    dict[str, pd.Series],
]:
    """
    Prepare a copy of the source chunk for staging insertion while
    retaining parsed-series metadata for rule evaluation.
    """
    prepared_chunk = raw_chunk.copy()

    parsed_datetime_series = {}

    for column_name in (
        DATETIME_SOURCE_COLUMNS
    ):
        parsed_series = pd.to_datetime(
            raw_chunk[column_name],
            errors="coerce",
        )

        parsed_datetime_series[
            column_name
        ] = parsed_series

        prepared_chunk[
            column_name
        ] = parsed_series


    for column_name in (
        NUMERIC_SOURCE_COLUMNS
    ):
        prepared_chunk[
            column_name
        ] = pd.to_numeric(
            raw_chunk[column_name],
            errors="coerce",
        )


    for column_name in (
        TEXT_SOURCE_COLUMNS
    ):
        prepared_chunk[
            column_name
        ] = raw_chunk[
            column_name
        ].astype("string")

    return (
        prepared_chunk,
        parsed_datetime_series,
    )


# ------------------------------------------
# Row-level failure-rule evaluation
# ------------------------------------------

def evaluate_failure_rules(
    raw_chunk: pd.DataFrame,
    prepared_chunk: pd.DataFrame,
    parsed_datetimes: dict[
        str,
        pd.Series,
    ],
) -> dict[str, pd.Series]:
    """
    Return one Boolean failure mask for each enabled row-level
    failure rule.
    """
    failure_masks = {}

    for rule_id, rule_definition in (
        ROW_LEVEL_FAILURE_RULES.items()
    ):
        rule_type = rule_definition[
            "rule_type"
        ]

        columns = rule_definition[
            "columns"
        ]

        parameters = rule_definition[
            "parameters"
        ]


        if rule_type == "NOT_NULL":
            column_name = columns[0]

            failure_mask = raw_chunk[
                column_name
            ].isna()


        elif rule_type == "NOT_BLANK":
            column_name = columns[0]

            column_series = raw_chunk[
                column_name
            ]

            failure_mask = (
                column_series.notna()
                & column_series.astype(
                    "string"
                ).str.strip().eq("")
            )


        elif rule_type == "ALLOWED_VALUES":
            column_name = columns[0]

            normalised_series = (
                raw_chunk[column_name]
                .astype("string")
                .str.strip()
            )

            allowed_values = parameters[
                "allowed_values"
            ]

            case_sensitive = (
                parameters.get(
                    "case_sensitive",
                    True,
                )
            )

            if case_sensitive:
                allowed_set = set(
                    allowed_values
                )

                failure_mask = (
                    normalised_series.notna()
                    & ~normalised_series.isin(
                        allowed_set
                    )
                )

            else:
                allowed_set = {
                    str(value).casefold()
                    for value
                    in allowed_values
                }

                failure_mask = (
                    normalised_series.notna()
                    & ~normalised_series
                    .str.casefold()
                    .isin(allowed_set)
                )


        elif rule_type == "DATETIME_PARSEABLE":
            column_name = columns[0]

            raw_series = raw_chunk[
                column_name
            ]

            parsed_series = (
                parsed_datetimes[
                    column_name
                ]
            )

            failure_mask = (
                raw_series.notna()
                & parsed_series.isna()
            )


        elif rule_type == "DATETIME_ORDER":
            start_column = columns[0]
            end_column = columns[1]

            parsed_start = (
                parsed_datetimes[
                    start_column
                ]
            )

            parsed_end = (
                parsed_datetimes[
                    end_column
                ]
            )

            failure_mask = (
                parsed_start.notna()
                & parsed_end.notna()
                & parsed_end.le(
                    parsed_start
                )
            )


        elif rule_type == "NUMERIC_RANGE":
            column_name = columns[0]

            raw_series = raw_chunk[
                column_name
            ]

            numeric_series = pd.to_numeric(
                raw_series,
                errors="coerce",
            )

            minimum_value = float(
                parameters["minimum"]
            )

            maximum_value = float(
                parameters["maximum"]
            )

            allow_null = parameters.get(
                "allow_null",
                True,
            )

            non_numeric_mask = (
                raw_series.notna()
                & numeric_series.isna()
            )

            below_minimum_mask = (
                numeric_series.notna()
                & numeric_series.lt(
                    minimum_value
                )
            )

            above_maximum_mask = (
                numeric_series.notna()
                & numeric_series.gt(
                    maximum_value
                )
            )

            failure_mask = (
                non_numeric_mask
                | below_minimum_mask
                | above_maximum_mask
            )

            if not allow_null:
                failure_mask = (
                    failure_mask
                    | raw_series.isna()
                )


        else:
            raise RuntimeError(
                "Unsupported row-level failure rule: "
                f"{rule_id} ({rule_type})"
            )


        failure_masks[rule_id] = (
            failure_mask
            .fillna(False)
            .astype(bool)
        )

    return failure_masks


# ------------------------------------------
# Build insertion SQL
# ------------------------------------------

STAGING_INSERT_COLUMNS = (
    ["staging_record_id"]
    + APPROVED_SOURCE_COLUMNS
    + [
        "source_file",
        "source_row_number",
        "pipeline_session_id",
        "quality_audit_session_id",
        "loaded_at",
    ]
)


STAGING_INSERT_SQL = sql.SQL(
    "INSERT INTO {}.{} ({}) VALUES %s"
).format(
    sql.Identifier("stg"),
    sql.Identifier(
        "cyclistic_rides"
    ),
    sql.SQL(", ").join(
        sql.Identifier(column_name)
        for column_name
        in STAGING_INSERT_COLUMNS
    ),
)


QUARANTINE_INSERT_COLUMNS = [
    "quarantine_id",
    "ride_id",
    "failed_rule",
    "reason",
    "source_file",
    "source_row_number",
    "raw_record",
    "pipeline_session_id",
    "quality_audit_session_id",
    "created_at",
]


QUARANTINE_INSERT_SQL = sql.SQL(
    "INSERT INTO {}.{} ({}) VALUES %s"
).format(
    sql.Identifier("quarantine"),
    sql.Identifier(
        "cyclistic_rides"
    ),
    sql.SQL(", ").join(
        sql.Identifier(column_name)
        for column_name
        in QUARANTINE_INSERT_COLUMNS
    ),
)


# ------------------------------------------
# Job recovery helper
# ------------------------------------------

def mark_job_failed(
    job_id: str,
    error_message: str,
    rows_read: int,
    rows_loaded: int,
    rows_quarantined: int,
    file_started_at: datetime,
) -> None:
    """
    Record FAILED status after the file transaction has rolled back.
    """
    recovery_connection = None
    recovery_cursor = None

    try:
        recovery_connection = (
            psycopg2.connect(
                host=DB_HOST,
                dbname=DB_NAME,
                user=DB_USER,
                password=DB_PASSWORD,
                port=DB_PORT,
                sslmode=DB_SSLMODE,
                connect_timeout=10,
            )
        )

        recovery_cursor = (
            recovery_connection.cursor()
        )

        completed_at = datetime.now(
            MELBOURNE_TIMEZONE
        )

        duration_seconds = round(
            (
                completed_at
                - file_started_at
            ).total_seconds(),
            3,
        )

        recovery_cursor.execute(
            """
            UPDATE audit.pipeline_job_log
            SET
                status = 'FAILED',
                completed_at = %s,
                rows_read = %s,
                rows_loaded = %s,
                rows_quarantined = %s,
                duration_seconds = %s,
                error_message = %s,
                updated_at = %s
            WHERE job_id = %s;
            """,
            (
                completed_at,
                rows_read,
                rows_loaded,
                rows_quarantined,
                duration_seconds,
                error_message[:10000],
                completed_at,
                job_id,
            ),
        )

        if recovery_cursor.rowcount != 1:
            raise RuntimeError(
                "FAILED job recovery update did not affect "
                "exactly one row."
            )

        recovery_connection.commit()

    finally:
        if recovery_cursor is not None:
            recovery_cursor.close()

        if recovery_connection is not None:
            recovery_connection.close()


# ------------------------------------------
# Initialise Part 4.4 outputs
# ------------------------------------------

DATABASE_FILE_LOAD_RESULTS = []
DATABASE_FILE_LOAD_FAILURES = []

DATABASE_TOTAL_ROWS_READ = 0
DATABASE_TOTAL_ROWS_LOADED = 0
DATABASE_TOTAL_SOURCE_ROWS_QUARANTINED = 0
DATABASE_TOTAL_QUARANTINE_FINDINGS = 0


log_database_event(
    message=(
        "DATABASE_FILE_LOADING_START: "
        f"ready_files="
        f"{len(DATABASE_READY_FILES)}; "
        f"chunk_size="
        f"{DB_LOAD_CHUNK_SIZE}; "
        f"insert_page_size="
        f"{DB_INSERT_PAGE_SIZE}."
    ),
    level="INFO",
)


# ------------------------------------------
# Process each source file independently
# ------------------------------------------

for load_plan_record in (
    DATABASE_READY_FILES
):
    source_file = (
        load_plan_record[
            "source_file"
        ]
    )

    source_path = Path(
        load_plan_record[
            "source_path"
        ]
    )

    planned_checksum = (
        load_plan_record[
            "file_checksum"
        ]
    )

    job_id = load_plan_record[
        "job_id"
    ]

    file_started_at = datetime.now(
        MELBOURNE_TIMEZONE
    )

    file_rows_read = 0
    file_rows_loaded = 0
    file_source_rows_quarantined = 0
    file_quarantine_findings = 0
    file_chunks_processed = 0

    connection = None
    cursor = None

    log_database_event(
        message=(
            "DATABASE_FILE_LOAD_START: "
            f"file={source_file}; "
            f"job_id={job_id}."
        ),
        level="INFO",
    )

    try:
        # ----------------------------------
        # Reconfirm stable file identity
        # ----------------------------------

        current_checksum = (
            calculate_file_sha256(
                source_path
            )
        )

        if current_checksum != planned_checksum:
            raise RuntimeError(
                "Source file checksum changed after Part 4.3. "
                f"Expected={planned_checksum}; "
                f"Observed={current_checksum}."
            )


        # ----------------------------------
        # Open file-level transaction
        # ----------------------------------

        connection = psycopg2.connect(
            host=DB_HOST,
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            port=DB_PORT,
            sslmode=DB_SSLMODE,
            connect_timeout=10,
        )

        connection.autocommit = False
        cursor = connection.cursor()


        # ----------------------------------
        # Lock and validate pending job
        # ----------------------------------

        cursor.execute(
            """
            SELECT
                status,
                source_file,
                file_checksum,
                pipeline_session_id
            FROM audit.pipeline_job_log
            WHERE job_id = %s
            FOR UPDATE;
            """,
            (job_id,),
        )

        job_record = cursor.fetchone()

        if job_record is None:
            raise RuntimeError(
                f"Job record does not exist: {job_id}"
            )

        (
            current_job_status,
            job_source_file,
            job_file_checksum,
            job_pipeline_session,
        ) = job_record


        if current_job_status not in {
            "PENDING",
            "RUNNING",
        }:
            raise RuntimeError(
                "Job is not eligible for loading. "
                f"status={current_job_status}."
            )


        if job_source_file != source_file:
            raise RuntimeError(
                "Job source filename does not match load plan."
            )


        if job_file_checksum != planned_checksum:
            raise RuntimeError(
                "Job checksum does not match load plan."
            )


        if (
            job_pipeline_session
            != DATABASE_PROCESSING_SESSION_ID
        ):
            raise RuntimeError(
                "Job pipeline session does not match "
                "the current database-processing session."
            )


        running_timestamp = datetime.now(
            MELBOURNE_TIMEZONE
        )

        cursor.execute(
            """
            UPDATE audit.pipeline_job_log
            SET
                status = 'RUNNING',
                started_at = %s,
                completed_at = NULL,
                error_message = NULL,
                updated_at = %s
            WHERE job_id = %s;
            """,
            (
                running_timestamp,
                running_timestamp,
                job_id,
            ),
        )


        # ----------------------------------
        # Verify source header
        # ----------------------------------

        source_header = pd.read_csv(
            source_path,
            nrows=0,
        )

        source_columns = list(
            source_header.columns
        )

        missing_approved_columns = sorted(
            set(APPROVED_SOURCE_COLUMNS)
            - set(source_columns)
        )

        unexpected_source_columns = sorted(
            set(source_columns)
            - set(APPROVED_SOURCE_COLUMNS)
        )


        if missing_approved_columns:
            raise RuntimeError(
                "Source file is missing approved column(s): "
                + ", ".join(
                    missing_approved_columns
                )
            )


        if unexpected_source_columns:
            raise RuntimeError(
                "Source file contains unapproved column(s): "
                + ", ".join(
                    unexpected_source_columns
                )
            )


        # ----------------------------------
        # Chunked read and routing
        # ----------------------------------

        source_rows_seen = 0

        chunk_iterator = pd.read_csv(
            source_path,
            usecols=APPROVED_SOURCE_COLUMNS,
            dtype=CSV_EXPLICIT_DTYPES or None,
            chunksize=DB_LOAD_CHUNK_SIZE,
            low_memory=False,
        )


        for raw_chunk in chunk_iterator:

            chunk_row_count = len(
                raw_chunk
            )

            if chunk_row_count == 0:
                continue


            source_row_numbers = pd.Series(
                range(
                    source_rows_seen + 2,
                    source_rows_seen
                    + chunk_row_count
                    + 2,
                ),
                index=raw_chunk.index,
                dtype="int64",
            )

            source_rows_seen += (
                chunk_row_count
            )

            file_rows_read += (
                chunk_row_count
            )

            file_chunks_processed += 1


            (
                prepared_chunk,
                parsed_datetimes,
            ) = prepare_chunk_types(
                raw_chunk=raw_chunk
            )


            failure_masks = (
                evaluate_failure_rules(
                    raw_chunk=raw_chunk,
                    prepared_chunk=(
                        prepared_chunk
                    ),
                    parsed_datetimes=(
                        parsed_datetimes
                    ),
                )
            )


            combined_failure_mask = (
                pd.Series(
                    False,
                    index=raw_chunk.index,
                    dtype=bool,
                )
            )

            for failure_mask in (
                failure_masks.values()
            ):
                combined_failure_mask = (
                    combined_failure_mask
                    | failure_mask
                )


            valid_mask = (
                ~combined_failure_mask
            )


            # ------------------------------
            # Prepare staging rows
            # ------------------------------

            staging_records = []

            loaded_at = datetime.now(
                MELBOURNE_TIMEZONE
            )


            for row_index in (
                prepared_chunk.index[
                    valid_mask
                ]
            ):
                row_values = [
                    str(uuid4())
                ]

                for column_name in (
                    APPROVED_SOURCE_COLUMNS
                ):
                    row_values.append(
                        python_safe_value(
                            prepared_chunk.at[
                                row_index,
                                column_name,
                            ]
                        )
                    )

                row_values.extend(
                    [
                        source_file,
                        int(
                            source_row_numbers.loc[
                                row_index
                            ]
                        ),
                        DATABASE_PROCESSING_SESSION_ID,
                        LOAD_QUALITY_AUDIT_SESSION_ID,
                        loaded_at,
                    ]
                )

                staging_records.append(
                    tuple(row_values)
                )


            if staging_records:
                execute_values(
                    cursor,
                    STAGING_INSERT_SQL.as_string(
                        connection
                    ),
                    staging_records,
                    page_size=(
                        DB_INSERT_PAGE_SIZE
                    ),
                )

                file_rows_loaded += len(
                    staging_records
                )


            # ------------------------------
            # Prepare quarantine findings
            # ------------------------------

            quarantine_records = []

            quarantined_row_indexes = set()


            for (
                rule_id,
                failure_mask,
            ) in failure_masks.items():

                failed_indexes = (
                    raw_chunk.index[
                        failure_mask
                    ]
                )

                if len(failed_indexes) == 0:
                    continue


                rule_definition = (
                    ROW_LEVEL_FAILURE_RULES[
                        rule_id
                    ]
                )

                rule_reason = (
                    rule_definition[
                        "description"
                    ]
                )


                for row_index in (
                    failed_indexes
                ):
                    quarantined_row_indexes.add(
                        row_index
                    )

                    raw_source_record = (
                        json_safe_record(
                            raw_chunk.loc[
                                row_index
                            ]
                        )
                    )

                    ride_id_value = (
                        raw_source_record.get(
                            "ride_id"
                        )
                    )

                    quarantine_records.append(
                        (
                            str(uuid4()),
                            (
                                None
                                if ride_id_value
                                is None
                                else str(
                                    ride_id_value
                                )
                            ),
                            rule_id,
                            rule_reason,
                            source_file,
                            int(
                                source_row_numbers.loc[
                                    row_index
                                ]
                            ),
                            Json(
                                raw_source_record
                            ),
                            DATABASE_PROCESSING_SESSION_ID,
                            LOAD_QUALITY_AUDIT_SESSION_ID,
                            datetime.now(
                                MELBOURNE_TIMEZONE
                            ),
                        )
                    )


            if quarantine_records:
                execute_values(
                    cursor,
                    QUARANTINE_INSERT_SQL.as_string(
                        connection
                    ),
                    quarantine_records,
                    page_size=(
                        DB_INSERT_PAGE_SIZE
                    ),
                )

                file_quarantine_findings += len(
                    quarantine_records
                )


            file_source_rows_quarantined += len(
                quarantined_row_indexes
            )


            log_database_event(
                message=(
                    "DATABASE_FILE_CHUNK_COMPLETE: "
                    f"file={source_file}; "
                    f"chunk={file_chunks_processed}; "
                    f"rows={chunk_row_count}; "
                    f"loaded="
                    f"{int(valid_mask.sum())}; "
                    f"quarantined_source_rows="
                    f"{len(quarantined_row_indexes)}; "
                    f"quarantine_findings="
                    f"{len(quarantine_records)}."
                ),
                level="INFO",
            )


        # ----------------------------------
        # Reconcile in-memory counts
        # ----------------------------------

        if (
            file_rows_read
            != file_rows_loaded
            + file_source_rows_quarantined
        ):
            raise RuntimeError(
                "File reconciliation failed before commit. "
                f"rows_read={file_rows_read}; "
                f"rows_loaded={file_rows_loaded}; "
                "source_rows_quarantined="
                f"{file_source_rows_quarantined}."
            )


        # ----------------------------------
        # Reconcile database counts
        # ----------------------------------

        cursor.execute(
            """
            SELECT COUNT(*)
            FROM stg.cyclistic_rides
            WHERE
                pipeline_session_id = %s
                AND source_file = %s;
            """,
            (
                DATABASE_PROCESSING_SESSION_ID,
                source_file,
            ),
        )

        database_staging_count = int(
            cursor.fetchone()[0]
        )


        cursor.execute(
            """
            SELECT
                COUNT(*) AS finding_count,
                COUNT(
                    DISTINCT source_row_number
                ) AS source_row_count
            FROM quarantine.cyclistic_rides
            WHERE
                pipeline_session_id = %s
                AND source_file = %s;
            """,
            (
                DATABASE_PROCESSING_SESSION_ID,
                source_file,
            ),
        )

        (
            database_quarantine_findings,
            database_quarantined_source_rows,
        ) = cursor.fetchone()

        database_quarantine_findings = int(
            database_quarantine_findings
        )

        database_quarantined_source_rows = int(
            database_quarantined_source_rows
        )


        if (
            database_staging_count
            != file_rows_loaded
        ):
            raise RuntimeError(
                "Database staging reconciliation failed. "
                f"expected={file_rows_loaded}; "
                f"observed={database_staging_count}."
            )


        if (
            database_quarantined_source_rows
            != file_source_rows_quarantined
        ):
            raise RuntimeError(
                "Database quarantine source-row reconciliation "
                "failed. "
                f"expected={file_source_rows_quarantined}; "
                "observed="
                f"{database_quarantined_source_rows}."
            )


        if (
            database_quarantine_findings
            != file_quarantine_findings
        ):
            raise RuntimeError(
                "Database quarantine finding reconciliation "
                "failed. "
                f"expected={file_quarantine_findings}; "
                f"observed={database_quarantine_findings}."
            )


        # ----------------------------------
        # Mark job successful
        # ----------------------------------

        file_completed_at = datetime.now(
            MELBOURNE_TIMEZONE
        )

        file_duration_seconds = round(
            (
                file_completed_at
                - file_started_at
            ).total_seconds(),
            3,
        )

        cursor.execute(
            """
            UPDATE audit.pipeline_job_log
            SET
                status = 'SUCCESS',
                completed_at = %s,
                rows_read = %s,
                rows_loaded = %s,
                rows_quarantined = %s,
                duration_seconds = %s,
                error_message = NULL,
                updated_at = %s
            WHERE
                job_id = %s
                AND status = 'RUNNING';
            """,
            (
                file_completed_at,
                file_rows_read,
                file_rows_loaded,
                file_source_rows_quarantined,
                file_duration_seconds,
                file_completed_at,
                job_id,
            ),
        )


        if cursor.rowcount != 1:
            raise RuntimeError(
                "SUCCESS job update did not affect exactly "
                "one RUNNING job record."
            )


        # ----------------------------------
        # Commit current file
        # ----------------------------------

        connection.commit()


        file_result = {
            "job_id": job_id,
            "source_file": source_file,
            "file_checksum": (
                planned_checksum
            ),
            "status": "SUCCESS",
            "chunks_processed": (
                file_chunks_processed
            ),
            "rows_read": (
                file_rows_read
            ),
            "rows_loaded": (
                file_rows_loaded
            ),
            "source_rows_quarantined": (
                file_source_rows_quarantined
            ),
            "quarantine_findings": (
                file_quarantine_findings
            ),
            "duration_seconds": (
                file_duration_seconds
            ),
            "completed_at": (
                file_completed_at.isoformat(
                    timespec="seconds"
                )
            ),
        }

        DATABASE_FILE_LOAD_RESULTS.append(
            file_result
        )

        DATABASE_TOTAL_ROWS_READ += (
            file_rows_read
        )

        DATABASE_TOTAL_ROWS_LOADED += (
            file_rows_loaded
        )

        DATABASE_TOTAL_SOURCE_ROWS_QUARANTINED += (
            file_source_rows_quarantined
        )

        DATABASE_TOTAL_QUARANTINE_FINDINGS += (
            file_quarantine_findings
        )


        log_database_event(
            message=(
                "DATABASE_FILE_LOAD_COMMITTED: "
                f"file={source_file}; "
                f"rows_read={file_rows_read}; "
                f"rows_loaded={file_rows_loaded}; "
                "source_rows_quarantined="
                f"{file_source_rows_quarantined}; "
                f"quarantine_findings="
                f"{file_quarantine_findings}; "
                f"duration_seconds="
                f"{file_duration_seconds}."
            ),
            level="INFO",
        )


    except (
        OperationalError,
        RuntimeError,
        ValueError,
        TypeError,
        OSError,
        pd.errors.ParserError,
        pd.errors.EmptyDataError,
        UnicodeDecodeError,
        psycopg2.DatabaseError,
    ) as error:

        if connection is not None:
            connection.rollback()


        failure_message = (
            f"{type(error).__name__}: "
            f"{error}"
        )


        try:
            mark_job_failed(
                job_id=job_id,
                error_message=(
                    failure_message
                ),
                rows_read=file_rows_read,
                rows_loaded=0,
                rows_quarantined=0,
                file_started_at=(
                    file_started_at
                ),
            )

        except Exception as recovery_error:
            log_database_event(
                message=(
                    "DATABASE_JOB_FAILURE_RECOVERY_FAILED: "
                    f"file={source_file}; "
                    f"job_id={job_id}; "
                    f"error={recovery_error}"
                ),
                level="CRITICAL",
            )


        failure_result = {
            "job_id": job_id,
            "source_file": source_file,
            "status": "FAILED",
            "rows_read_before_rollback": (
                file_rows_read
            ),
            "error_type": (
                type(error).__name__
            ),
            "error_message": str(error),
            "failed_at": datetime.now(
                MELBOURNE_TIMEZONE
            ).isoformat(
                timespec="seconds"
            ),
        }

        DATABASE_FILE_LOAD_FAILURES.append(
            failure_result
        )


        log_database_event(
            message=(
                "DATABASE_FILE_LOAD_ROLLED_BACK: "
                f"file={source_file}; "
                f"job_id={job_id}; "
                f"rows_read_before_rollback="
                f"{file_rows_read}; "
                f"error_type="
                f"{type(error).__name__}; "
                f"error={error}"
            ),
            level="ERROR",
        )


    finally:
        if cursor is not None:
            cursor.close()

        if connection is not None:
            connection.close()


# ------------------------------------------
# Build Part 4.4 summary
# ------------------------------------------

if DATABASE_FILE_LOAD_FAILURES:
    DATABASE_FILE_LOAD_STATUS = (
        "PARTIAL_FAILURE"
        if DATABASE_FILE_LOAD_RESULTS
        else "FAILED"
    )
else:
    DATABASE_FILE_LOAD_STATUS = (
        "SUCCESS"
    )


DATABASE_FILE_LOAD_SUMMARY = {
    "database_processing_session_id": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "quality_audit_session_id": (
        LOAD_QUALITY_AUDIT_SESSION_ID
    ),
    "status": (
        DATABASE_FILE_LOAD_STATUS
    ),
    "files_ready": len(
        DATABASE_READY_FILES
    ),
    "files_loaded_successfully": len(
        DATABASE_FILE_LOAD_RESULTS
    ),
    "files_failed": len(
        DATABASE_FILE_LOAD_FAILURES
    ),
    "total_rows_read": (
        DATABASE_TOTAL_ROWS_READ
    ),
    "total_rows_loaded": (
        DATABASE_TOTAL_ROWS_LOADED
    ),
    "total_source_rows_quarantined": (
        DATABASE_TOTAL_SOURCE_ROWS_QUARANTINED
    ),
    "total_quarantine_findings": (
        DATABASE_TOTAL_QUARANTINE_FINDINGS
    ),
    "file_results": deepcopy(
        DATABASE_FILE_LOAD_RESULTS
    ),
    "file_failures": deepcopy(
        DATABASE_FILE_LOAD_FAILURES
    ),
    "completed_at": datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds"),
}


log_database_event(
    message=(
        "DATABASE_FILE_LOADING_COMPLETE: "
        f"status={DATABASE_FILE_LOAD_STATUS}; "
        "files_loaded="
        f"{len(DATABASE_FILE_LOAD_RESULTS)}; "
        "files_failed="
        f"{len(DATABASE_FILE_LOAD_FAILURES)}; "
        "rows_read="
        f"{DATABASE_TOTAL_ROWS_READ}; "
        "rows_loaded="
        f"{DATABASE_TOTAL_ROWS_LOADED}; "
        "source_rows_quarantined="
        f"{DATABASE_TOTAL_SOURCE_ROWS_QUARANTINED}; "
        "quarantine_findings="
        f"{DATABASE_TOTAL_QUARANTINE_FINDINGS}."
    ),
    level=(
        "INFO"
        if DATABASE_FILE_LOAD_STATUS
        == "SUCCESS"
        else "WARNING"
    ),
)


# ------------------------------------------
# Display Part 4.4 summary
# ------------------------------------------

print(
    "\n========== Database File Loading "
    "==========\n"
)

print(
    f"Database-processing session : "
    f"{DATABASE_PROCESSING_SESSION_ID}"
)
print(
    f"Quality-audit session       : "
    f"{LOAD_QUALITY_AUDIT_SESSION_ID}"
)
print(
    f"Load status                 : "
    f"{DATABASE_FILE_LOAD_STATUS}"
)
print(
    f"Files ready                 : "
    f"{len(DATABASE_READY_FILES)}"
)
print(
    f"Files loaded successfully   : "
    f"{len(DATABASE_FILE_LOAD_RESULTS)}"
)
print(
    f"Files failed                : "
    f"{len(DATABASE_FILE_LOAD_FAILURES)}"
)
print(
    f"Rows read                   : "
    f"{DATABASE_TOTAL_ROWS_READ:,}"
)
print(
    f"Rows loaded                 : "
    f"{DATABASE_TOTAL_ROWS_LOADED:,}"
)
print(
    f"Source rows quarantined     : "
    f"{DATABASE_TOTAL_SOURCE_ROWS_QUARANTINED:,}"
)
print(
    f"Quarantine findings         : "
    f"{DATABASE_TOTAL_QUARANTINE_FINDINGS:,}"
)


print("\nFile results:")

for result in DATABASE_FILE_LOAD_RESULTS:
    print(
        f"  {result['source_file']:<40} "
        f"SUCCESS  "
        f"read={result['rows_read']:,} "
        f"loaded={result['rows_loaded']:,} "
        "quarantined="
        f"{result['source_rows_quarantined']:,}"
    )


for failure in DATABASE_FILE_LOAD_FAILURES:
    print(
        f"  {failure['source_file']:<40} "
        f"FAILED   "
        f"{failure['error_type']}: "
        f"{failure['error_message']}"
    )


print("\n" + "=" * 82)

if DATABASE_FILE_LOAD_STATUS == "SUCCESS":
    print(
        "✓ All eligible files were committed successfully."
    )

elif DATABASE_FILE_LOAD_STATUS == "PARTIAL_FAILURE":
    print(
        "⚠ Some files committed successfully and some were rolled back."
    )

else:
    print(
        "✗ No eligible source file completed successfully."
    )

print("✓ File-level transaction boundaries were enforced.")
print("✓ Failure rows were routed to quarantine.")
print("✓ Warning-only records remained loadable.")
print("✓ Job-log records were updated.")
print("✓ Ready for Part 4.5 reconciliation and finalisation.")
print("=" * 82)